# Class 2 - Modern CNN Architectures: From Classic to SOTA

> **Course: Computer Vision — Advanced Module**  
> **Topics:** Top CNN Architectures · Object Detection · Attention Mechanisms · Transfer Learning

---

This notebook provides a deep dive into the **state-of-the-art convolutional neural network (CNN) architectures** and key trends in modern computer vision. Each section combines theory with working PyTorch code.


# Table of Contents

0. [Environment Setup](#0-setup)
1. [Key concepts in SOTA CNN architectures](#1-key-concepts)
   - 1.1 Internal Covariate Shift — and why the modern explanation is different
   - 1.2 Normalization Layers — Batch, Layer and Group Norm
   - 1.3 Depthwise + Pointwise convolutions
   - 1.4 Activation Functions
2. [Top Performing CNN Architectures](#2-architectures)
   - 2.1 ConvNeXt
   - 2.2 ConvNeXt V2 — Global Response Normalization (GRN)
   - 2.3 EfficientNetV2
   - 2.4 NFNet (Normalizer-Free Networks)
   - 2.5 ResNeXt & Wide Residual Networks
3. [SOTA Object Detection & Segmentation](#3-detection)
   - 3.1 Detection metrics: IoU, Precision/Recall, AP and mAP
   - 3.2 Faster R-CNN (RPN, RoI Pooling → RoIAlign)
   - 3.3 YOLO Series (v5 → YOLO26)
   - 3.4 Mask R-CNN
   - 3.5 EfficientDet
4. [Key Trends and Characteristics](#4-key-trends)
   - 4.1 Attention Mechanisms (SE, CBAM)
   - 4.2 Efficiency vs Performance: FLOPs Analysis
   - 4.3 1D-CNNs for Temporal Signal Analysis
5. [Transfer Learning: Fine-Tuning a Pretrained CNN](#5-transfer-learning)
   - 5.1 Fine-tuning strategies
   - 5.2 Hands on — fine-tuning ConvNeXt-Tiny on CIFAR-10
6. [📚 Summary](#6-summary)
7. [🔗 References](#7-references)

---

<a name='0-setup'></a>
# ⚙️ 0. Environment Setup

In [ ]:
# Install required libraries
#
# Colab already ships torch, torchvision, timm, matplotlib and numpy, and its
# torch/torchvision wheels are matched to the runtime's CUDA driver — reinstalling
# them from PyPI is slow and can break the GPU runtime. Install only what is missing.
#
# NOTE: do *not* use `uv pip install` here. Outside a virtualenv uv exits with
# "No virtual environment found", and `!` never raises, so the notebook would happily
# continue and die later on the import. If you want uv on Colab, it is
# `!pip install -q uv && uv pip install --system -q <pkgs>`.
!pip install -q hjson

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import timm
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from collections import OrderedDict

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'🔧 PyTorch version : {torch.__version__}')
print(f'🔧 Torchvision     : {torchvision.__version__}')
print(f'🔧 TIMM version    : {timm.__version__}')
print(f'🖥️  Device          : {device}')

<a name='1-key-concepts'></a>

# 1. Key concepts in SOTA CNN architectures
We are introducing new concepts to discuss state-of-the-art CNN architectures.

## 1.1 *Internal Covariate Shift*
It's the problem that BatchNorm was invented to solve.

#### *The intuition*
Imagine you're training layer 5 of a deep network. Layer 5 learns to transform its inputs in a useful way. But layer 4 is also learning and changing its weights — which means the distribution of inputs arriving at layer 5 keeps shifting during training.
<p>
Layer 5 is essentially trying to hit a moving target. Example :
</p>

```text
Epoch 1:   Layer 4 outputs  →  mean≈0.5,  std≈1.2  → Layer 5 adapts
Epoch 2:   Layer 4 outputs  →  mean≈1.8,  std≈0.4  → Layer 5 has to re-adapt
Epoch 3:   Layer 4 outputs  →  mean≈-0.3, std≈2.1  → Layer 5 has to re-adapt again
              ↑
         This constant distribution shift = Internal Covariate Shift
```

<p>
<img src="https://raw.githubusercontent.com/EzequielMatiasArevalo/tuia-computer-vision/refs/heads/main/media/pictures/Class_2_Modern_CNN/internal-cov-shift.webp">
</p>
The word "covariate" comes from statistics — it just means input features. "Internal" means it's happening inside the network, between layers, not just at the input.

#### *Why it's a problem*

1. Slows down training
Each layer wastes capacity constantly readjusting to the new input distribution instead of learning the actual task.
2. Requires very small learning rates
Large learning rates cause big weight updates → big distribution shifts → downstream layers destabilize → gradients explode or vanish.
3. Saturating activations
If inputs drift into the saturation zone of sigmoid or tanh, gradients become nearly zero

---

#### ⚠️ Careful: this is the *historical* motivation, not the current explanation

Everything above is the original 2015 argument by Ioffe & Szegedy, and it is worth knowing because
it explains the *shape* of the algorithm (subtract a mean, divide by a std, learn γ and β back).
But it is no longer the accepted explanation of **why** BatchNorm works.

[Santurkar et al. (NeurIPS 2018)](https://arxiv.org/abs/1805.11604) tested the claim directly: they
injected **deliberate** distributional noise after every BatchNorm layer — reintroducing internal
covariate shift on purpose — and the networks still trained just as fast. Their conclusion is that
"distributional stability of layer inputs has little to do with the success of BatchNorm".

The accepted explanation today is that BN **smooths the optimization landscape**: it makes the loss
and its gradients change more slowly and more predictably as the weights move, which is what allows
the larger learning rates and the faster, more stable convergence we actually observe.

> **Exam-safe answer:** BatchNorm was *motivated* by internal covariate shift, but it *works* by
> smoothing the loss landscape.

In [ ]:
x_normal = torch.linspace(-3, 3, 100)    # well-distributed inputs
x_shifted = torch.linspace(3, 9, 100)    # shifted inputs (covariate shift)

sigmoid = torch.sigmoid

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(x_normal, sigmoid(x_normal), color='#2ecc71', linewidth=2)
axes[0].fill_between(x_normal, 0,
                     sigmoid(x_normal) * (1 - sigmoid(x_normal)),
                     alpha=0.3, color='#2ecc71', label='gradient region')
axes[0].set_title('Normal distribution → healthy gradients', fontsize=11)
axes[0].set_xlabel('Input'); axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(x_shifted, sigmoid(x_shifted), color='#e74c3c', linewidth=2)
axes[1].fill_between(x_shifted, 0,
                     sigmoid(x_shifted) * (1 - sigmoid(x_shifted)),
                     alpha=0.3, color='#e74c3c', label='gradient region (≈0!)')
axes[1].set_title('Shifted distribution → vanishing gradients', fontsize=11)
axes[1].set_xlabel('Input'); axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Internal Covariate Shift — Saturation Problem', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 1.2 - Normalization Layers

### 1.2.1 Batch Normalization


**Batch normalization** is a technique used in training neural networks to stabilize and speed up learning by normalizing the inputs of each layer.

---
<b>How it works</b>

For each mini-batch during training:

1. Compute the mean and variance of the layer’s inputs:
<p>
<img src="https://raw.githubusercontent.com/EzequielMatiasArevalo/tuia-computer-vision/refs/heads/main/media/pictures/Class_2_Modern_CNN/norm-layer.png">
</p>

2. Normalize the inputs:
<p>
<img src="https://raw.githubusercontent.com/EzequielMatiasArevalo/tuia-computer-vision/refs/heads/main/media/pictures/Class_2_Modern_CNN/normalization.png">
</p>
3. Apply learnable scaling and shifting:
<p>
<img src="https://raw.githubusercontent.com/EzequielMatiasArevalo/tuia-computer-vision/refs/heads/main/media/pictures/Class_2_Modern_CNN/norm-linear.png">
</p>

* **μ (mean)** and **σ² (variance)** come from the batch
* **ε** is a small constant for numerical stability
* **γ (gamma)** and **β (beta)** are learned parameters

---
<p>Why it’s useful</b>

* Reduces internal covariate shift (inputs changing distribution during training)
* Allows higher learning rates
* Speeds up convergence
* Acts as a regularizer (can reduce need for dropout)

---
<p>During inference</b>

Instead of batch statistics, it uses **running averages** of mean and variance computed during training.

---
<p>Where it’s used</b>

* Common in CNNs and deep networks
* Typically placed **after linear/conv layers and before activation**

---
<p>Intuition</b>

It keeps activations in a stable range, preventing layers from constantly adapting to shifting input distributions.

---
<p>Problems</b>:

- ❌ Needs a large enough batch to compute stable statistics
- ❌ Training vs inference discrepancy (running mean/var)
- ❌ Complicated in distributed training (sync across GPUs)
- ❌ Doesn't work with batch size = 1


### 1.2.2 Layer Normalization

Layer normalization is a normalization technique used in neural networks that normalizes the activations across the features of a single sample, instead of across a batch.



| Aspect                         | Batch Normalization    | Layer Normalization          |
| ------------------------------ | ---------------------- | ---------------------------- |
| Normalization over             | Batch (across samples) | Features (within one sample) |
| Depends on batch size          | Yes                    | No                           |
| Works well with                | CNNs                   | RNNs, Transformers           |
| Training vs inference behavior | Different              | Same                         |

In the layout below the **batch index B runs along a row** and the **channel index C runs down a
column**. Read the arrows against that layout — this is exactly where the two get mixed up:

```text
         B0    B1    B2    B3          ← batch dimension (one column per sample)
C0     [  x     x     x     x  ]   ← BatchNorm: one channel, all samples
C1     [  x     x     x     x  ]        (µ, σ over N, H, W → along this row)
C2     [  x     x     x     x  ]
C3     [  x     x     x     x  ]
          ↑
          LayerNorm: one sample, all channels
          (µ, σ over C, H, W → down this column)
```

So BatchNorm slices **horizontally** (fix a channel, sweep the batch) and LayerNorm slices
**vertically** (fix a sample, sweep the channels) — which is also what the table above and the
Group Norm table in 1.2.3 say.

### 1.2.3 Group Normalization

**Group Normalization** (GN), introduced by [Wu & He (2018)](https://arxiv.org/pdf/1803.08494), is a normalization layer meant as a **batch-size–agnostic** alternative to Batch Normalization.

Where BN aggregates statistics over the **batch** (and spatial locations), GN **never uses the batch dimension**: it partitions channels into **G groups** and normalizes using the mean and variance computed **within each group**, over that group’s channels and spatial positions **(H × W)** for a single sample.

#### Motivation (from the paper)

- BN’s estimated batch mean and variance become **noisy when the mini-batch is small**, which hurts accuracy; this matters for **detection, segmentation, and video**, where memory often forces small batches.

- GN’s computation is **independent of batch size**, so training behaves similarly for batch size 2 or 32.

- On **ImageNet with ResNet-50**, GN had about **10.6% lower top-1 error than BN when the batch size was 2**; with **typical large batches**, GN was **comparable to BN** and **better than other normalization variants** tested in that setting.

- GN **transfers cleanly** from pre-training to fine-tuning and worked well on **COCO** (detection/segmentation) and **Kinetics** (video) in their experiments.

#### How it differs from BN and Layer Norm

| Method | Main axes used for μ, σ² (per channel group / sample) | Uses batch? |
|--------|--------------------------------------------------------|-------------|
| **Batch Norm** | N × H × W (per channel) | Yes |
| **Layer Norm** | C × H × W (per sample) | No |
| **Group Norm** | (C/G) × H × W **per group**, per sample | No |

Like BN, GN still applies **learnable scale γ and shift β** after normalizing, so the layer can undo normalization if the network needs it.

#### Practical notes

- **Hyperparameter G**: the number of groups must divide the channel count **C**. A common default in the paper and in libraries is **G = 32** when **C** is divisible by 32.
- **PyTorch**: `nn.GroupNorm(num_groups, num_channels)` — same API at train and inference (no running buffers like BN).

#### Illustration (grouped channels per normalization region)

<p>
<img src="https://raw.githubusercontent.com/EzequielMatiasArevalo/tuia-computer-vision/refs/heads/main/media/pictures/Class_2_Modern_CNN/group-norm.png">
</p>

**Reference:** Yuxin Wu & Kaiming He, *Group Normalization*, ECCV 2018 / [arXiv:1803.08494](https://arxiv.org/pdf/1803.08494).

### 1.2.4 Summary of Normalization Layers

The Group Norm paper defines all four layers with a single formula and changes only **which set of
elements `S_i` is averaged over**. Quoting section 3.1 of [Wu & He (2018)](https://arxiv.org/pdf/1803.08494),
in the paper's own order:

**Batch Norm** — the set is

```
S_i = { k | k_C = i_C }
```

where `i_C` (and `k_C`) denotes the sub-index of `i` (and `k`) along the `C` axis. This means that
the pixels sharing the same channel index are normalized together, i.e., for each channel, BN
computes µ and σ along the **(N, H, W)** axes.

**Layer Norm** — the set is

```
S_i = { k | k_N = i_N }
```

meaning that LN computes µ and σ along the **(C, H, W)** axes, for each sample.

**Instance Norm** — the set is

```
S_i = { k | k_N = i_N , k_C = i_C }
```

meaning that IN computes µ and σ along the **(H, W)** axes, for each sample and each channel.

**Group Norm** sits between LN and IN: it splits the `C` channels into `G` groups and computes µ and
σ over **(C/G, H, W)** per group, per sample. `G = 1` recovers Layer Norm and `G = C` recovers
Instance Norm.

| Layer | Averaged over | Uses the batch? |
|---|---|---|
| **Batch Norm** | N, H, W — per channel | ✅ yes |
| **Layer Norm** | C, H, W — per sample | ❌ no |
| **Instance Norm** | H, W — per sample, per channel | ❌ no |
| **Group Norm** | C/G, H, W — per group, per sample | ❌ no |

> Note the index sets are easy to misread: `k_C = i_C` ("same channel") is **BatchNorm's**, and
> `k_N = i_N` ("same sample") is **LayerNorm's**.

<p>
<img src="https://raw.githubusercontent.com/EzequielMatiasArevalo/tuia-computer-vision/refs/heads/main/media/pictures/Class_2_Modern_CNN/normalizations-draw.png">
</p>


## 1.3 Depthwise + Pointwise convolutions
**Depthwise convolution** is a variant of standard convolution used in CNNs that processes each input channel **independently**, instead of mixing all channels together.

---

#### Standard Convolution (baseline)

In a normal convolution:

* Input: H×W×C
* Filters:K×K×C
* Each filter spans **all channels**
* Output channels = number of filters

👉 Each filter mixes spatial + cross-channel information at once.

<p>
<img src="https://raw.githubusercontent.com/EzequielMatiasArevalo/tuia-computer-vision/refs/heads/main/media/pictures/Class_2_Modern_CNN/std-conv.webp">
</p>

---

#### Depthwise Convolution

In **depthwise convolution**:

* You use **one filter per input channel**
* Each filter is K×K×1
* No mixing between channels

If input has (C) channels:

* You apply **C separate filters**
* Output also has (C) channels


---

#### How it works (step-by-step)

For an input with 3 channels (RGB):

* Apply 1 filter to Red channel
* Apply 1 filter to Green channel
* Apply 1 filter to Blue channel

Each channel is convolved **independently**.


---

#### Depthwise Separable Convolution

Depthwise is usually **not used alone**.

It’s combined with:

* **Depthwise convolution** → spatial filtering
* **Pointwise (1×1) convolution** → channel mixing

This combination is called:
👉 **Depthwise Separable Convolution**

<p>
<img src="https://raw.githubusercontent.com/EzequielMatiasArevalo/tuia-computer-vision/refs/heads/main/media/pictures/Class_2_Modern_CNN/depthwise.webp">
</p>


Used in:

* MobileNet
* EfficientNet
* Xception

---

#### Why it matters

#### Advantages

* Much **fewer parameters**
* Lower **compute cost (FLOPs)**
* Ideal for:

  * Mobile devices
  * Edge AI
  * Real-time inference

#### Trade-off

* Slightly less expressive than full convolution
* Needs pointwise conv to recover performance

---

#### Intuition (simple analogy)

* Standard convolution = mixing all colors while painting
* Depthwise = painting each color separately
* Pointwise = blending colors afterward

---

#### Summary

| Feature        | Standard Conv         | Depthwise Conv        |
| -------------- | --------------------- | --------------------- |
| Channel mixing | Yes                   | No                    |
| Filters        | K x K x C             | K x K x 1             |
| Cost           | High                  | Low                   |
| Use alone      | Yes                   | Rare                  |
| Typical usage  | General CNNs          | Efficient CNNs        |

---

If needed, can show PyTorch / TensorFlow implementation or compare with grouped convolution (closely related concept).


In [ ]:


C = 96   # channels

# Standard conv: every output channel sees ALL input channels
standard_conv = nn.Conv2d(
    in_channels  = C,
    out_channels = C,
    kernel_size  = 7,
    padding      = 3,
    # groups       = 1     # default — all channels connected
)

# Depthwise conv: each output channel sees ONLY its own input channel
depthwise_conv = nn.Conv2d(
    in_channels  = C,
    out_channels = C,
    kernel_size  = 7,
    padding      = 3,
    groups       = C     # ← groups=C makes it depthwise
)

x   = torch.randn(3, C, 56, 56)
out = depthwise_conv(x)
out_std = standard_conv(x)

print(f'Input  : {x.shape}')
print(f'Output : {out.shape}')   # same shape — only spatial mixing
print(f'Output standard conv: {out_std.shape}')   # same shape — only spatial mixing
# Parameter comparison
std_params = sum(p.numel() for p in standard_conv.parameters())
dw_params  = sum(p.numel() for p in depthwise_conv.parameters())

print(f'\nStandard Conv params : {std_params:,}')
print(f'Depthwise Conv params: {dw_params:,}')
print(f'Reduction factor     : {std_params / dw_params:.0f}×')


---
## 1.4 Activation Functions

The TOC promises this section, the architectures in section 2 depend on it, and it is the one
"free" design change in a CNN: swapping the activation costs no parameters and almost no FLOPs.

Every activation below is a scalar function applied element-wise. What separates them is
**smoothness** (is the function differentiable everywhere?), **whether negatives survive** (can a
unit output a negative value, or die at zero?), and **self-gating** (does the function multiply the
input by a soft, input-dependent gate?).

| Activation | Definition | Smooth? | Negatives? | Where you meet it in this class |
|---|---|---|---|---|
| **ReLU** | `max(0, x)` | No (kink at 0) | Killed | ResNet, ResNeXt, WideResNet |
| **Leaky ReLU / PReLU** | `x if x>0 else a·x` (`a` fixed / learned) | No | Leaked | GAN discriminators, older detectors |
| **GELU** | `x · Φ(x)`, Φ = Gaussian CDF | Yes | Soft, bounded below | ViT, **ConvNeXt** (§2.1) |
| **SiLU / Swish** | `x · σ(x)` | Yes | Soft, bounded below | **EfficientNet(V2)** (§2.3), **YOLO** (§3.3) |
| **Mish** | `x · tanh(softplus(x))` | Yes | Soft, bounded below | YOLOv4-era backbones |

#### Why the field moved off ReLU

- **Dying ReLU.** A unit whose pre-activation is negative for every input in the dataset outputs
  exactly 0 and receives exactly 0 gradient — forever. Leaky ReLU was the first cheap fix.
- **Smoothness.** GELU, SiLU and Mish are differentiable everywhere and have non-monotone behaviour
  for small negatives, which empirically gives a better-conditioned loss landscape — the same
  argument you just saw for BatchNorm in 1.1.
- **Self-gating.** SiLU is `x` multiplied by a *soft gate* `σ(x)` in [0, 1]. This is the same idea as
  the channel attention of Squeeze-and-Excitation in §4.1, applied at the level of a single scalar
  instead of a whole channel.

#### The design point that matters for section 2

ConvNeXt does not just swap ReLU for GELU — it also uses **one activation per block** instead of one
after every convolution, copying the Transformer block layout. Fewer, smoother non-linearities;
same accuracy budget. That is the pattern to remember: modern CNNs borrowed the Transformer's
*block structure*, and the activation change came along with it.

#### In PyTorch

```python
nn.ReLU(inplace=True)   # ResNet family
nn.LeakyReLU(0.1)
nn.GELU()               # ConvNeXt, ViT
nn.SiLU(inplace=True)   # EfficientNet, YOLO — also called Swish
nn.Mish()
```

All five are already used in the code cells further down: look for `nn.GELU()` in the ConvNeXt block
(§2.1) and `nn.SiLU()` in MBConv (§2.3) and C2f (§3.3).

---
<a name='2-architectures'></a>
# 2. Top Performing CNN Architectures

The landscape of CNN architectures has evolved dramatically. These modern designs borrow heavily from the success of **Vision Transformers (ViTs)** while maintaining the computational efficiency of convolutional operations.


## 2.1 ConvNeXt

<p>
<img src="https://raw.githubusercontent.com/EzequielMatiasArevalo/tuia-computer-vision/refs/heads/main/media/pictures/Class_2_Modern_CNN/convexnet.jpeg">
</p>

**ConvNeXt** (Liu et al., 2022 — Meta AI) is a modernized ResNet that systematically adopted every successful design decision from Vision Transformers:

### 🔑 Key Innovation: The ConvNeXt Block

| Design Choice | ResNet | ViT (Vision Transformers )| ConvNeXt |
|---|---|---|---|
| Stem / patchify | 7×7 conv, stride 2 + 3×3 max-pool, stride 2 — overlapping, 4× downsample | 16×16 conv, stride 16 — non-overlapping | **4×4 conv, stride 4 — "patchify", non-overlapping, 4× downsample** |
| Activation | ReLU | GELU | **GELU** |
| Normalization | BatchNorm | LayerNorm | **LayerNorm** |
| Convolution | 3×3 | Attention | **7×7 Depthwise** |
| Stage ratio | (3,4,6,3) | - | **(3,3,9,3)** |



## 2.2 ConvNeXt V2 - Global Response Normalization (GRN)

---
**ConvNeXt V2** (Woo et al., 2023) added **Global Response Normalization (GRN)** — a layer that suppresses redundant feature activations, further closing the gap with masked autoencoders.
**GRN** stands for **Global Response Normalization**.

It is a normalization technique introduced in **ConvNeXt V2** to improve how feature maps are scaled and balanced across channels.

---

#### Core idea

GRN normalizes features based on their **global (spatial) response**, and then **recalibrates channels to compete with each other**.

<p>
<img src="https://raw.githubusercontent.com/EzequielMatiasArevalo/tuia-computer-vision/refs/heads/main/media/pictures/Class_2_Modern_CNN/convnextv2.png" width="500" heigh="500">
</p>

#### Why GRN is useful :

1. <b>Encourages channel competition</b>

  - Strong channels don’t dominate
  - Weak channels get boosted


2. <b>Prevents feature collapse</b>

  - Especially important in **self-supervised learning**
  - Keeps representations diverse


3. <b>Works well with vision tasks</b>

  - Captures **global context**
  - Improves robustness and generalization

---

#### GRN vs other normalizations

| Method       | Normalizes over       | Key effect                      |
| ------------ | --------------------- | ------------------------------- |
| BatchNorm    | Batch                 | Stabilizes training             |
| LayerNorm    | Channels              | Transformer-style normalization |
| InstanceNorm | Spatial (per channel) | Style invariance                |
| **GRN**      | Global + channels     | Channel competition             |



**GRN = normalize global channel strength and use it to rebalance feature maps dynamically.**



In [ ]:
class ConvNeXtBlock(nn.Module):
    """
    ConvNeXt Block — the core building unit.

    Key design choices vs classic ResNet block:
      • 7×7 depthwise conv  (large receptive field, like self-attention)
      • Inverted bottleneck  (expand → GELU → compress)
      • LayerNorm instead of BatchNorm
      • GELU instead of ReLU
      • Fewer activations / normalizations in the block
    """
    def __init__(self, dim: int, layer_scale: float = 1e-6):
        super().__init__()
        self.block = nn.Sequential(
            # Step 1: Depthwise 7×7 conv (spatial mixing per channel)
            nn.Conv2d(dim, dim, kernel_size=7, padding=3, groups=dim),
            # Step 2: Permute to (B, H, W, C) for LayerNorm on channel dim
        )
        self.norm  = nn.LayerNorm(dim, eps=1e-6)
        # Step 3: Inverted bottleneck with pointwise (linear) layers
        self.pwconv1 = nn.Linear(dim, 4 * dim)    # expand
        self.act     = nn.GELU()
        self.pwconv2 = nn.Linear(4 * dim, dim)    # compress
        # Learnable per-channel scale (improves training stability)
        self.gamma = nn.Parameter(
            layer_scale * torch.ones(dim), requires_grad=True
        ) if layer_scale > 0 else None

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = x
        x = self.block(x)                         # (B, C, H, W)
        x = x.permute(0, 2, 3, 1)                # → (B, H, W, C)
        x = self.norm(x)
        x = self.pwconv1(x)
        x = self.act(x)
        x = self.pwconv2(x)
        if self.gamma is not None:
            x = self.gamma * x
        x = x.permute(0, 3, 1, 2)                # → (B, C, H, W)
        return residual + x                        # Residual connection


# ── Quick test ──────────────────────────────────────────────────────────────
block = ConvNeXtBlock(dim=96).to(device)
x     = torch.randn(3, 96, 56, 56).to(device)    # (Batch=2, C=96, H=56, W=56)
out   = block(x)

print(f'ConvNeXt Block')
print(f'  Input  shape : {x.shape}')
print(f'  Output shape : {out.shape}')
params = sum(p.numel() for p in block.parameters())
print(f'  Parameters   : {params:,}')

In [ ]:
# ── Load pretrained ConvNeXt via timm ────────────────────────────────────────
# timm has the full model family: tiny, small, base, large, xlarge
convnext = timm.create_model('convnext_xlarge', pretrained=False, num_classes=10)

print('=== ConvNeXt-XLarge Architecture (timm) ===')
print(f'Model stages: {[name for name, _ in convnext.named_children()]}')

# Parameter count per stage
total = 0
print('\n📊 Parameters per stage:')
for name, module in convnext.named_children():
    stage_params = sum(p.numel() for p in module.parameters())
    total += stage_params
    print(f'  {name:<20} → {stage_params:>8,} params')
print(f'  {"TOTAL":<20} → {total:>8,} params')

# Forward pass
dummy = torch.randn(1, 3, 224, 224)
with torch.no_grad():
    logits = convnext(dummy)
print(f'\n✅ Output logits shape: {logits.shape}  (batch=1, num_classes=10)')

### 2.2.1 Hands on - Testing a pretrained `convnext_xlarge` on ImageNet-1k.
We are downloading a smaller version of the ImageNet dataset. Since the pre-trained model was trained on the original ImageNet-1k, we need to use the original label mapping rather than the labels inferred from this dataset's directory structure (one folder per class).

> ⚠️ **Download size.** The Kaggle archive below is **~7.7 GB (539,826 images)** and the unzipped copy
> needs roughly the same again. On a Colab runtime this takes several minutes and a large slice of the
> disk quota. If you are short on space or time, unzip only a handful of class folders
> (`unzip -q ./data/imagenet-256.zip 'imagenet-256/n015*' -d ./data/imagenet-256`) — the demo below
> works just as well on a subset.

*Q 2.2.1*: Why? What's happened with the Head of the model ?

In [ ]:
! mkdir -p ./data
! [ -f ./data/imagenet-256.zip ] || curl -L -o ./data/imagenet-256.zip https://www.kaggle.com/api/v1/datasets/download/dimensi0n/imagenet-256
! [ -d ./data/imagenet-256 ] || unzip -q ./data/imagenet-256.zip -d ./data/imagenet-256
! curl https://gist.githubusercontent.com/yrevar/942d3a0ac09ec9e5eb3a/raw/238f720ff059c1f82f368259d1ca4ffa5dd8f9f5/imagenet1000_clsidx_to_labels.txt > ./data/imagenet-256/clsidx.json

### 2.2.2 - Load Dataset

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
import hjson
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

ROOT = './data/imagenet-256'   # folder containing train/ and val/
CLASSES = hjson.load(open(f'{ROOT}/clsidx.json'))
BATCH_SIZE = 30

val_transform = transforms.Compose([
    transforms.CenterCrop(224),             # center 224 crop from 256
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std =[0.229, 0.224, 0.225]
    ),
])

val_dataset = ImageFolder(
    root      = f'{ROOT}/',
    transform = val_transform
)

print(f'Val samples   : {len(val_dataset):,}')
print(f'First 5 classes: {val_dataset.classes[:5]}')
print(f'Original classes: {[ c.split(',')[0] for c in CLASSES.values()][:5]}')
print(f'Total classes : {len(val_dataset.classes)}')

# shuffle=True (with a fixed seed) — with shuffle=False the single batch we display below
# would be 30 consecutive images of the *same* class, which tells us nothing about the model.
val_loader = DataLoader(
    val_dataset,
    batch_size  = BATCH_SIZE,
    shuffle     = True,
    generator   = torch.Generator().manual_seed(0),
    num_workers = 8,
    pin_memory  = True,
    persistent_workers = True
)

# Quick sanity check
images, labels = next(iter(val_loader))
print(f'\nBatch shape : {images.shape}')    # ( BATCH_SIZE, 3, 224, 224)
print(f'Labels shape: {labels.shape}')     # (BATCH_SIZE,)
print(f'Pixel range : [{images.min():.2f}, {images.max():.2f}]')

### 2.2.3 - Loading convnext models

We are using two ConvNeXt models:

- One fully pre-trained
- Another overwriting its classification head to 512 classes without any fine-tuning.

<b>

Q: What do you think the output will be for both models?

We will compare their performance and see the results.
</b>

In [ ]:
from torch.utils.data import DataLoader
convnext_pt = timm.create_model('convnext_xlarge', pretrained=True, num_classes=len(val_dataset.classes)).to(device).eval()
convnext_pt_512 = timm.create_model('convnext_xlarge', pretrained=True, num_classes=512).to(device).eval()

# images, targets = next(iter(loader))
images = images.to(device)
targets = labels.to(device)
with torch.no_grad():
    logits = convnext_pt(images)

with torch.no_grad():
    logits_512 = convnext_pt_512(images)

pred = logits.argmax(dim=1)
pred_512 = logits_512.argmax(dim=1)

print('=== convnext_xlarge (ImageNet backbone ')
print(f'  logits shape : {logits.shape}')

for i in range(min(4, images.size(0))):
    ti, pi = targets[i].item(), pred[i].item()
    predicted_class = CLASSES[f"{pi}"]
    inference_class = val_dataset.classes[ti]
    print(f'    [{i}] true=[ {targets[i]:<12} | {inference_class} ] pred=[ {pred[i]:<12} | {predicted_class} ]')

print("="*50)
print("=== covnext with head of 512 classes === ")

for i in range(min(4, images.size(0))):
    ti, pi = targets[i].item(), pred_512[i].item()
    predicted_class = CLASSES[f"{pi}"]
    inference_class = val_dataset.classes[ti]
    print(f'    [{i}] true=[ {targets[i]:<12} | {inference_class} ] pred=[ {pred_512[i]:<12} | {predicted_class} ]')

# ── Visualize batch
# (denormalize ImageNet stats for display) ─────────────────


### 2.2.4 - We denormalize the images to be displayed
We normalize the images to be displayed and define a function to display a batch of images.

In [ ]:
mean = torch.tensor([0.485, 0.456, 0.406], device=images.device).view(1, 3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225], device=images.device).view(1, 3, 1, 1)
vis = (images * std + mean).clamp(0, 1).cpu()

In [ ]:
def normalize_label(name: str) -> str:
    """Make the two label vocabularies comparable.

    ImageFolder derives class names from directory names ('academic_gown', 'afghan_hound')
    while the clsidx gist uses the human-readable form ('academic gown', 'Afghan hound').
    Comparing them raw marks ~44% of *correct* predictions as wrong.
    """
    return name.lower().replace('_', ' ').strip()


def display_batch(images, targets, preds, description="", max_show=16):
    MAX_SHOW = max_show

    n_show = images.size(0)
    ncols = min(4, MAX_SHOW)
    nrows = (MAX_SHOW + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(1.2 * ncols, 1.2 * nrows))
    axes = np.atleast_1d(axes).ravel()

    for i in range(n_show):
        if i >= MAX_SHOW:
            break
        ax = axes[i]
        ax.imshow(images[i].permute(1, 2, 0).numpy())

        ti, pi = targets[i], preds[i]

        predicted_class = CLASSES[f"{pi}"].split(',')[0].strip()
        inference_class = val_dataset.classes[ti]

        ok = normalize_label(predicted_class) == normalize_label(inference_class)
        title = f"true: {inference_class}\npred: {predicted_class}"
        ax.set_title(title, fontsize=9, color='darkgreen' if ok else 'darkred')
        ax.axis('off')
    for j in range(n_show, len(axes)):
        axes[j].axis('off')
    plt.suptitle(f"{description}", fontsize=11, y=1.02)
    plt.tight_layout()
    plt.show()

In [ ]:
display_batch(vis, targets, pred, "Pretrained model")
print("="*50)
display_batch(vis, targets, pred_512, "Head without Finetune" )


In [ ]:

del convnext_pt
del convnext_pt_512
torch.cuda.empty_cache() if torch.cuda.is_available() else None

*A 2.2.1*: Because the model’s output logits correspond to the fixed 1,000 classes and their specific index ordering used during training.

If we rely on directory-based labels, the class indices will not align with the model’s expected label space, leading to incorrect predictions and evaluation mismatches.

---
## 2.3 EfficientNetV2

**EfficientNetV2** (Tan & Le, 2021 — Google) builds upon EfficientNet by solving two limitations:
1. **Slow training speed** at high resolutions
2. **Depthwise convolutions are hardware-unfriendly** at early stages


<p>
<img src="https://raw.githubusercontent.com/EzequielMatiasArevalo/tuia-computer-vision/refs/heads/main/media/pictures/Class_2_Modern_CNN/efficient-net.png">
</p>

#### Key Innovations: what actually changed from V1 to V2

| What | EfficientNet (V1) | EfficientNetV2 |
|---|---|---|
| **Early-stage block** | MBConv everywhere | **Fused-MBConv** in the early stages (MBConv kept later) |
| **Image size during training** | Fixed | **Progressive learning**: 128 → 192 → 224 over training |
| **Regularization / augmentation** | Fixed strength | **Progressive**: grows together with the image size |
| **NAS objective** | FLOPs + accuracy | FLOPs + accuracy + **training speed** |

Two things are worth separating, because the original version of this table conflated them:

- **Fused-MBConv** is an *architectural* change, and it is what the NAS search space was extended with.
- **Progressive learning** is a *training-schedule* change, not an architectural one. It is the
  observation that small images need weak regularization and large images need strong regularization,
  so ramping both together is faster than fixing either.

**Fused-MBConv** replaces the depthwise 3×3 + pointwise 1×1 expansion with a single **regular 3×3
conv + pointwise conv**. It costs more FLOPs on paper but is faster in wall-clock time on modern
accelerators (TPUs, GPUs with tensor cores), because depthwise convolutions are memory-bound and do
not use tensor cores well. This is why V2 uses it only in the early stages, where the spatial
resolution is high and the channel count is low.

#### Kept from V1

| Component | Status in V2 |
|---|---|
| SE block (Squeeze-and-Excitation) | Kept — only in the MBConv stages |
| SiLU activation | Kept throughout |
| Compound scaling of depth/width/resolution | Kept, but with a cap on the max image size |

Result:
  - ✅ Up to 4× faster training than V1
  - ✅ Up to 6.8× smaller than V1 at comparable accuracy
  - ✅ SOTA ImageNet accuracy **at the time (2021)** — see the caveat under the benchmark chart at the
    end of the notebook before comparing these numbers with ConvNeXt's
  - ✅ Better parameter efficiency

In [ ]:
class MBConv(nn.Module):
    """Mobile Inverted Bottleneck (MBConv) — used in EfficientNet."""
    def __init__(self, in_channels: int, out_channels: int, expand_ratio: int = 4, stride: int = 1):
        super().__init__()
        hidden = in_channels * expand_ratio
        self.use_residual = (stride == 1 and in_channels == out_channels)
        self.block = nn.Sequential(
            # Pointwise expand
            nn.Conv2d(in_channels, hidden, 1, bias=False),
            nn.BatchNorm2d(hidden), nn.SiLU(),
            # Depthwise 3×3
            nn.Conv2d(hidden, hidden, 3, stride=stride, padding=1, groups=hidden, bias=False),
            nn.BatchNorm2d(hidden), nn.SiLU(),
            # Pointwise project
            nn.Conv2d(hidden, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels),
        )

    def forward(self, x):
        out = self.block(x)
        return out + x if self.use_residual else out


class FusedMBConv(nn.Module):
    """
    Fused MBConv — key innovation in EfficientNetV2.
    Fuses the depthwise + pointwise-expand into a SINGLE 3×3 conv.
    This is more hardware-efficient at early stages.
    """
    def __init__(self, in_channels: int, out_channels: int, expand_ratio: int = 4, stride: int = 1):
        super().__init__()
        hidden = in_channels * expand_ratio
        self.use_residual = (stride == 1 and in_channels == out_channels)
        self.block = nn.Sequential(
            # Single fused 3×3 conv (replaces DWConv + pointwise-expand)
            nn.Conv2d(in_channels, hidden, 3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(hidden), nn.SiLU(),
            # Pointwise project
            nn.Conv2d(hidden, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels),
        )

    def forward(self, x):
        out = self.block(x)
        return out + x if self.use_residual else out


# ── Compare FLOPs: MBConv vs FusedMBConv ────────────────────────────────────
def count_params(model):
    return sum(p.numel() for p in model.parameters())

C = 32  # channels
mb      = MBConv(C, C)
fused   = FusedMBConv(C, C)
x       = torch.randn(1, C, 56, 56)

print('📐 MBConv vs Fused-MBConv')
print(f'  MBConv      — params: {count_params(mb):>6,}  output: {mb(x).shape}')
print(f'  FusedMBConv — params: {count_params(fused):>6,}  output: {fused(x).shape}')
print('\n  FusedMBConv removes the depthwise step → fewer kernel launches,')
print('  better GPU utilization in early low-channel stages.')

In [ ]:
# ── EfficientNetV2-S via timm ────────────────────────────────────────────────
effv2 = timm.create_model('efficientnetv2_s', pretrained=False, num_classes=1000)

# Show which stages use FusedMBConv vs MBConv
print('=== EfficientNetV2-S — Block Distribution ===')
fused_count = mb_count = other_count = 0
for name, module in effv2.named_modules():
    cls = type(module).__name__
    if 'Fused' in cls: fused_count += 1
    elif 'InvertedResidual' in cls or 'MBConv' in cls: mb_count += 1

total_params = count_params(effv2)
print(f'  Total parameters : {total_params/1e6:.2f}M')
print(f'\n  EfficientNetV2-S uses:')
print(f'  • Fused-MBConv in early stages  (stages 0-2)  → faster training')
print(f'  • MBConv       in later stages  (stages 3-6)  → parameter efficient')

dummy = torch.randn(1, 3, 224, 224)
with torch.no_grad():
    out = effv2(dummy)
print(f'\n  Output: {out.shape}')

---
## 2.4 NFNet — Normalizer-Free Networks

The summary table at the end of this notebook has a row for NFNet and the benchmark chart plots an
NFNet-F0 point, so here is the section that earns them.

**NFNet** ([Brock et al., 2021 — DeepMind](https://arxiv.org/abs/2102.06171)) asks the obvious
follow-up to section 1.2: *if BatchNorm has all those drawbacks, can we simply delete it?* The
answer is yes — but only if you replace the three things BN was quietly doing for you.

#### Why remove BatchNorm at all

BN's problems are the ones listed in 1.2.1, and they get worse at scale:

- it is **batch-size dependent**, so small batches (detection, segmentation, video) give noisy statistics;
- it behaves **differently at train and inference** (running averages), a recurring source of bugs;
- it **breaks the independence between training examples** in a batch, which leaks information and
  makes some contrastive/distributed setups subtly wrong;
- it is **expensive**: it adds memory traffic and forces synchronization across GPUs.

#### The three replacements

| BN was providing | NFNet's replacement |
|---|---|
| Well-scaled activations | **Scaled Weight Standardization** — normalize the *convolution weights* (subtract the mean, divide by the std of each filter) instead of the activations. No batch is involved and train == inference. |
| Variance control through depth | **Signal-propagation-preserving residual blocks**: `x_{l+1} = x_l + α · f(x_l / β_l)`, with the scalars `α` and `β_l` chosen analytically so the activation variance grows in a known, controlled way. |
| Implicit regularization + tolerance of large learning rates | **Adaptive Gradient Clipping (AGC)** — clip each parameter's gradient by the **ratio** `‖G‖ / ‖W‖` rather than by an absolute threshold, so the clipping scales with the size of the weights it applies to. AGC is what makes normalizer-free training stable at large batch sizes. |

#### Practical notes

- The family is **NFNet-F0 … F6** (F0 is the point in the benchmark chart at the end of this notebook).
- In `timm`: `timm.create_model('dm_nfnet_f0', pretrained=True)` — the `dm_` prefix marks the weights
  ported from the DeepMind release.
- Trade-off to remember: NFNets remove BN's batch dependence and train fast on TPU/GPU pods, but they
  are **not** cheap in parameters or FLOPs relative to ConvNeXt/EfficientNetV2 at similar accuracy.
  Their contribution is conceptual — normalization is not load-bearing, *good signal propagation* is.

> Connect this back to 1.1: Santurkar et al. said BN works by smoothing the landscape rather than by
> fixing covariate shift. NFNet is the constructive version of that claim — get the smooth landscape
> by construction (weight standardization + scaled residuals + AGC) and you no longer need BN.

---
## 2.5 ResNeXt

### 2.5.1  Wide Residual Networks

These are evolution of the classic ResNet that improve accuracy by rethinking **width** and **cardinality** (number of independent paths), rather than just adding more layers.

#### The Three Axes of Scaling (He et al.):
``` text
          Depth         Width         Cardinality
         (layers)     (channels)      (groups)
ResNet      ✅            ❌              ❌
WideResNet  ❌            ✅              ❌
ResNeXt     ❌            ❌              ✅
```



### 2.5.2 Grouped Convolutions

Instead of one wide transform, apply `C` independent parallel transforms (cardinality) and sum them. This is equivalent to group convolution and maps cleanly to the Inception idea.

`ResNeXt-50 (32×4d)` = cardinality 32, group width 4

In [ ]:
class ResNetBottleneck(nn.Module):
    """Classic ResNet Bottleneck: 1×1 → 3×3 → 1×1."""
    expansion = 4

    def __init__(self, in_ch, mid_ch, stride=1):
        super().__init__()
        out_ch = mid_ch * self.expansion
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, mid_ch, 1, bias=False), nn.BatchNorm2d(mid_ch), nn.ReLU(inplace=True),
            nn.Conv2d(mid_ch, mid_ch, 3, stride=stride, padding=1, bias=False), nn.BatchNorm2d(mid_ch), nn.ReLU(inplace=True),
            nn.Conv2d(mid_ch, out_ch, 1, bias=False), nn.BatchNorm2d(out_ch),
        )
        self.shortcut = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False), nn.BatchNorm2d(out_ch)
        ) if in_ch != out_ch else nn.Identity()
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.relu(self.block(x) + self.shortcut(x))


class ResNeXtBottleneck(nn.Module):
    """
    ResNeXt Bottleneck: same as ResNet but the 3×3 conv uses GROUPS.

    cardinality (C) = number of independent groups = "cardinality"
    base_width      = channels per group

    This achieves better accuracy at the same parameter count by
    leveraging more diverse feature transformations.
    """
    expansion = 2  # simplified for demo

    def __init__(self, in_ch, mid_ch, stride=1, cardinality=32, base_width=4):
        super().__init__()
        # Grouped channel width
        width   = int(mid_ch * (base_width / 64.0)) * cardinality
        out_ch  = mid_ch * self.expansion
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, width, 1, bias=False), nn.BatchNorm2d(width), nn.ReLU(inplace=True),
            # ← KEY: groups=cardinality (grouped convolution)
            nn.Conv2d(width, width, 3, stride=stride, padding=1,
                      groups=cardinality, bias=False),
            nn.BatchNorm2d(width), nn.ReLU(inplace=True),
            nn.Conv2d(width, out_ch, 1, bias=False), nn.BatchNorm2d(out_ch),
        )
        self.shortcut = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False), nn.BatchNorm2d(out_ch)
        ) if in_ch != out_ch else nn.Identity()
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.relu(self.block(x) + self.shortcut(x))


class WideResNetBlock(nn.Module):
    """
    Wide ResNet Block: fewer layers but WIDER channels (width_factor k).
    WRN-28-10 = 28 layers, width factor 10 → channels × 10.
    """
    def __init__(self, in_ch, out_ch, stride=1, dropout=0.3):
        super().__init__()
        self.block = nn.Sequential(
            nn.BatchNorm2d(in_ch), nn.ReLU(inplace=True),
            nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Dropout(p=dropout),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
        )
        self.shortcut = nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False) if in_ch != out_ch else nn.Identity()

    def forward(self, x):
        return self.block(x) + self.shortcut(x)


# ── Compare parameter efficiency ─────────────────────────────────────────────
in_ch = 256
resnet_block  = ResNetBottleneck(in_ch, 64)
resnext_block = ResNeXtBottleneck(in_ch, 128, cardinality=32, base_width=4)
wide_block    = WideResNetBlock(in_ch, 512)

x = torch.randn(2, 256, 28, 28)
print('📊 Architecture Comparison — Single Block')
print(f'  {"Block":<20} {"Params":>10}  {"Output shape"}')
print('  ' + '-'*50)
for name, block in [('ResNet-Bottleneck', resnet_block),
                     ('ResNeXt-Bottleneck', resnext_block),
                     ('WideResNet-Block',   wide_block)]:
    out = block(x)
    p   = count_params(block)
    print(f'  {name:<20} {p:>10,}  {out.shape}')

In [ ]:
# ── Visualize Grouped Convolution (the heart of ResNeXt) ─────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

def draw_conv_diagram(ax, title, n_groups, in_ch=8, out_ch=8, color='steelblue'):
    ax.set_xlim(0, 10); ax.set_ylim(0, 10); ax.axis('off')
    ax.set_title(title, fontsize=13, fontweight='bold', pad=10)
    ch_per_group = in_ch // n_groups
    colors = plt.cm.Set2(np.linspace(0, 1, n_groups))
    w = 8.0 / n_groups
    for g in range(n_groups):
        c = colors[g]
        x0 = 1 + g * w
        # Input block
        ax.add_patch(mpatches.FancyBboxPatch((x0, 7), w*0.85, 1.2, boxstyle='round,pad=0.05', fc=c, ec='k', lw=1.5, alpha=0.8))
        # Output block
        ax.add_patch(mpatches.FancyBboxPatch((x0, 1.5), w*0.85, 1.2, boxstyle='round,pad=0.05', fc=c, ec='k', lw=1.5, alpha=0.8))
        # Arrow
        cx = x0 + w*0.425
        ax.annotate('', xy=(cx, 2.7), xytext=(cx, 7.0),
                    arrowprops=dict(arrowstyle='->', color=c, lw=2))
        ax.text(cx, 4.85, f'G{g+1}', ha='center', va='center', fontsize=9, color=c,
                bbox=dict(fc='white', ec=c, boxstyle='round', lw=1.5, pad=0.3))
    ax.text(5, 9.0, f'Input ({in_ch} ch)', ha='center', va='center', fontsize=11,
            bbox=dict(fc='lightyellow', ec='gray', boxstyle='round', pad=0.4))
    ax.text(5, 0.5, f'Output ({out_ch} ch)', ha='center', va='center', fontsize=11,
            bbox=dict(fc='lightcyan', ec='gray', boxstyle='round', pad=0.4))
    ax.text(5, 5.8, f'{n_groups} independent group(s)', ha='center', fontsize=9, style='italic', color='gray')

draw_conv_diagram(axes[0], 'Standard Conv\n(1 group = all channels connected)', n_groups=1)
draw_conv_diagram(axes[1], 'Grouped Conv (ResNeXt)\n(4 groups = parallel paths)', n_groups=4)

plt.suptitle('Grouped Convolution: The Heart of ResNeXt', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('grouped_conv.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Grouped convolutions split channels into independent groups,')
print('   increasing model capacity (cardinality) without adding parameters.')

<a name='3-detection'></a>
# 3. SOTA Object Detection & Segmentation

## 3.1 Detection metrics: IoU, Precision/Recall, AP and mAP

Before any architecture: **how do we know a detector is good?** Every model in this section is
ranked by **mAP**, IoU appears three cells from now as if it were common knowledge, and neither is
defined anywhere in the notebook. Fixing that first makes the rest of section 3 readable.

### 3.1.1 IoU — Intersection over Union

IoU measures how well two boxes overlap:

$$\text{IoU}(A, B) = \frac{|A \cap B|}{|A \cup B|} \in [0, 1]$$

0 means disjoint, 1 means identical. It is scale-invariant — a 10-pixel error on a 20-pixel object
and a 10-pixel error on a 500-pixel object give very different IoUs, which is exactly what we want
from a localization measure.

```python
import torch
from torchvision.ops import box_iou

a = torch.tensor([[10., 10., 110., 110.]])   # xyxy
b = torch.tensor([[20., 20., 120., 120.]])
print(box_iou(a, b))                          # -> tensor([[0.6807]])   # 8100 / 11900
```

IoU is used in **three different places**, and it is worth keeping them apart:

1. **Assignment during training** (§3.2.1: anchors with IoU > 0.7 are positive, < 0.3 negative);
2. **NMS**, to decide which overlapping predictions are duplicates;
3. **Evaluation**, as the threshold that decides whether a prediction counts as a hit.

### 3.1.2 From IoU to Precision and Recall

Fix a class and an IoU threshold `t` (COCO's classic single-threshold report uses `t = 0.5`). Sort
every prediction for that class by confidence, then walk down the list:

| | Meaning |
|---|---|
| **TP** | The prediction has IoU ≥ `t` with a ground-truth box **that has not already been matched** |
| **FP** | IoU < `t` with every ground-truth box, **or** it is a duplicate of an already-matched one |
| **FN** | A ground-truth box that no prediction matched |

$$\text{Precision} = \frac{TP}{TP + FP} \qquad \text{Recall} = \frac{TP}{TP + FN}$$

The "already matched" rule is what makes NMS matter: without it, ten copies of the same correct box
would count as ten true positives.

### 3.1.3 AP — Average Precision

Sweeping the confidence threshold from high to low traces out a **precision–recall curve** for that
class. **AP is the area under that curve.** Lower the threshold and recall goes up while precision
usually goes down; AP summarizes that whole trade-off in one number, so you never have to pick an
operating point to compare two models.

COCO computes it as the **mean of the (interpolated) precision at 101 recall levels**
`[0.00, 0.01, …, 1.00]`, not as a raw trapezoidal integral.

### 3.1.4 mAP — and the thing everybody gets wrong

**mAP is AP averaged over classes.** COCO-style mAP additionally averages over **10 IoU thresholds**,
`0.50, 0.55, …, 0.95` — that is what `mAP@[.50:.95]` (often written just `mAP` or `AP`) means.

> ⚠️ **AP is never computed per image and then averaged.** Detections from *all* images are pooled
> into a single precision–recall curve **per class**. A per-image average will not match any
> published number.

Common variants you will see in papers and in the Ultralytics output:

| Symbol | What it averages |
|---|---|
| `AP50` / `mAP@0.5` | Classes, at IoU 0.50 only — the lenient, "did you find it" number |
| `AP75` | Classes, at IoU 0.75 — the strict-localization number |
| `mAP@[.50:.95]` | Classes **and** the 10 IoU thresholds — the headline COCO metric |
| `APs / APm / APl` | The same, restricted to small / medium / large objects |

### 3.1.5 Computing it yourself

```python
# pip install torchmetrics
from torchmetrics.detection import MeanAveragePrecision

metric = MeanAveragePrecision(box_format='xyxy', iou_type='bbox')
# preds  : list of dicts with 'boxes' (N,4), 'scores' (N,), 'labels' (N,)
# targets: list of dicts with 'boxes' (M,4), 'labels' (M,)
metric.update(preds, targets)
print(metric.compute())    # -> map, map_50, map_75, map_small, map_medium, map_large, ...
```

The hands-on in §3.2.3 below only *draws* boxes above a fixed score threshold and prints a count —
which tells you nothing about quality. Re-running it over a labelled set with `MeanAveragePrecision`
is the natural exercise, and it is the only way to compare the detectors in 3.2–3.5 honestly.

**Reference:** the definitive implementation is COCO's
[`cocoeval.py`](https://github.com/cocodataset/cocoapi/blob/master/PythonAPI/pycocotools/cocoeval.py).

## 3.2 Faster R-CNN
*It's a two-stage detector, not a backbone.*

<p>
<img src="https://raw.githubusercontent.com/EzequielMatiasArevalo/tuia-computer-vision/refs/heads/main/media/pictures/Class_2_Modern_CNN/fastercnn.png" width="300" height="300">
</p>



Faster R-CNN is easiest to read as three parts stacked on top of each other, each one replacing a
slower predecessor.

**1. Backbone + FPN — look at the image once.**
The image passes through a CNN backbone a single time. A **Feature Pyramid Network** taps that
backbone at several depths and emits feature maps `P2…P5`: different spatial resolutions, the same
channel width. Small objects are found on the fine maps, large ones on the coarse maps. This is the
whole point of "Faster" — the original R-CNN ran the CNN once *per region proposal*.

**2. Region Proposal Network — propose regions inside the network.**
Earlier detectors got their candidate regions from an external algorithm (Selective Search, Edge
Boxes) that knew nothing about the task. Faster R-CNN's RPN is a small convolutional head that
slides over the pyramid, scores a set of **anchor boxes** at every location as object/background, and
regresses four offsets to tighten each one. Because it is trained jointly with the detector, the
proposals are tuned to your data — and they are essentially free, since they reuse the backbone
features that are already computed. Details in §3.2.1.

**3. RoI head — classify and refine each proposal.**
Each surviving proposal is cropped out of the pyramid into a fixed-size feature patch (§3.2.2) and
sent through a small head that outputs a class distribution over `K + 1` categories (the `+ 1` is
background) plus a per-class box refinement. Training minimizes a multi-task loss: cross-entropy for
the class, Smooth L1 on the encoded box deltas — and the same two terms again for the RPN.

#### In PyTorch

You do not assemble any of this by hand for a standard setup — `torchvision` ships the whole
detector, which is what the hands-on cell below uses:

```python
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights

weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT     # COCO-pretrained
model = fasterrcnn_resnet50_fpn(weights=weights).eval()
```

To fine-tune it on your own classes you replace only the box predictor, keeping everything else:

```python
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)   # num_classes counts background
```

And to swap the **backbone** — the point the original text was trying to make — you wrap any CNN in
an FPN and hand it to the generic `FasterRCNN` class:

```python
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.backbone_utils import resnet_fpn_backbone

backbone = resnet_fpn_backbone(backbone_name='resnet18', weights='IMAGENET1K_V1', trainable_layers=3)
model = FasterRCNN(backbone, num_classes=num_classes)
```

For a non-torchvision backbone (a `timm` ConvNeXt, for instance) the equivalent is
`torchvision.models.detection.backbone_utils.BackboneWithFPN`, which takes any feature-extracting
module plus the names and channel counts of the layers to tap. That is how you put the architectures
from section 2 underneath a section-3 detector.

<p>
<img src="https://raw.githubusercontent.com/EzequielMatiasArevalo/tuia-computer-vision/refs/heads/main/media/pictures/Class_2_Modern_CNN/fastrcnn.png">
</p>

### 3.2.1 Region Proposal Network (RPN)

The RPN slides over the feature map produced by the backbone (e.g., ResNet + FPN) and, at each spatial location, asks:

"Is there an object here, and if so, where exactly?"

---

### *Anchors Boxes*

At each feature map cell, the RPN places k anchor boxes of different scales and aspect ratios (e.g., 3 scales × 3 ratios = 9 anchors per location).

### Two heads on top of the RPN
For each anchor, the RPN predicts:

|      Head     | Output                       | What it does |
| --- | --- | --- |
|Classification |2 scores (object / background)| Is there any object here? |
|Regression      |4 deltas (Δx, Δy, Δw, Δh)     | Refine the anchor box to tightly fit the object |


---

### *Training signal*

- Anchors with IoU > 0.7 with a GT box → positive (object)
- Anchors with IoU < 0.3 → negative (background)
- RPN loss = L_cls (binary CE) + λ · L_reg (Smooth L1)

---

### *NMS ( Non-Maximum Suppression ) to filter proposals*
After scoring all anchors, Non-Maximum Suppression keeps the top-N proposals (e.g., 2000 during training, 300 at inference) before passing them downstream.
~20k raw anchors → RPN scoring → NMS → ~300 region proposals



### 3.2.2 RoI Pooling — and why the code actually runs RoIAlign
*The problem*: region proposals have different sizes, but the classification head (FC layers) needs a fixed-size input.
RoI Pooling solves this by extracting a fixed H×W feature patch from the shared feature map for each proposal, regardless of its original size.
How it works
Given a proposal of size h × w on the feature map and a target output of H × W (e.g., 7×7):

- Divide the proposal region into an H × W grid of sub-windows
- Max-pool within each sub-window → one value per cell
- Output: always a 7 × 7 × C tensor ✅

```text
Feature Map
┌─────────────────────────────┐
│                             │
│   ┌──────────────┐          │
│   │   Proposal   │          │  ← irregular size (h × w)
│   │   region     │          │
│   └──────────────┘          │
└─────────────────────────────┘
          │
          ▼  Divide into 7×7 grid, max-pool each cell
     ┌─┬─┬─┬─┬─┬─┬─┐
     ├─┼─┼─┼─┼─┼─┼─┤
     ├─┼─┼─┼─┼─┼─┼─┤  → Fixed 7×7×C tensor
     ├─┼─┼─┼─┼─┼─┼─┤
     └─┴─┴─┴─┴─┴─┴─┘
```

---

#### RoIAlign — what `fasterrcnn_resnet50_fpn` really uses

RoI Pooling as described above **quantizes twice**: once when the proposal's floating-point
coordinates are snapped to the integer feature-map grid, and again when that region is split into
`H × W` sub-windows whose borders are also rounded. On a stride-16 feature map, rounding a coordinate
by half a cell is **8 pixels** in the input image. For box classification that is survivable; for a
28×28 instance mask it is not, which is why Mask R-CNN (§3.4) introduced the fix.

**RoIAlign removes both roundings.** It keeps the exact floating-point proposal coordinates, samples
a fixed number of points inside each sub-window (typically 4) at their exact real-valued positions
using **bilinear interpolation** of the four neighbouring feature cells, and then averages or
max-pools those samples. No `floor`, no snapping — the operation is continuous in the box
coordinates, so gradients flow back to them cleanly.

```text
RoI Pooling                          RoIAlign
─────────────                        ─────────────
box  → round to grid  (loses up      box  → kept as float
        to ½ cell twice)             sample points → bilinear interp
max-pool integer cells               average the sampled values
```

This is not a footnote: **the code you run in §3.2.3 uses RoIAlign, not RoI Pooling.**
`torchvision`'s `fasterrcnn_resnet50_fpn` builds its RoI head with

```python
from torchvision.ops import MultiScaleRoIAlign
box_roi_pool = MultiScaleRoIAlign(featmap_names=['0', '1', '2', '3'], output_size=7, sampling_ratio=2)
```

`MultiScaleRoIAlign` adds one more thing on top of RoIAlign: with an FPN there are several feature
maps, so it first **picks which pyramid level** each proposal should be cropped from based on the
proposal's size (big boxes → coarse maps, small boxes → fine maps), then applies RoIAlign there.

You can call the bare operator directly:

```python
from torchvision.ops import roi_align
patches = roi_align(feature_map, boxes, output_size=(7, 7), spatial_scale=1/16, sampling_ratio=2)
```

**Takeaway:** RoI Pooling is the concept, RoIAlign is the implementation everything modern actually
ships. When the summary table at the end of this notebook credits Mask R-CNN with "RoIAlign", this
is the operation it means.

### 3.2.3 Hands on - Faster R-CNN + COCO-style detection


Run **COCO-pretrained Faster R-CNN (ResNet-50-FPN)** on a sample image to **draw boxes and labels** (80 COCO thing classes + background).

> Note that drawing boxes above a fixed score threshold and counting them is a *demo*, not an > evaluation. To say anything about quality you need the mAP machinery from §3.1 over a labelled set.


In [ ]:
import io
import urllib.request

from PIL import Image
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights

weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
coco_detector = fasterrcnn_resnet50_fpn(weights=weights).to(device).eval()
coco_names = weights.meta['categories']
preprocess = weights.transforms()

IMAGE_URL = 'https://ultralytics.com/images/bus.jpg'
raw = urllib.request.urlopen(IMAGE_URL, timeout=30).read()
pil = Image.open(io.BytesIO(raw)).convert('RGB')
img_tensor = preprocess(pil).to(device)
with torch.no_grad():
    det_out = coco_detector([img_tensor])[0]

SCORE_THR = 0.5
keep = det_out['scores'] >= SCORE_THR
boxes = det_out['boxes'][keep].cpu()
scores = det_out['scores'][keep].cpu()
labels = det_out['labels'][keep].cpu()

fig, ax = plt.subplots(1, 1, figsize=(12, 8))
ax.imshow(pil)
for box, sc, lab in zip(boxes, scores, labels):
    x1, y1, x2, y2 = box.tolist()
    rect = mpatches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor='lime', linewidth=2)
    ax.add_patch(rect)
    name = coco_names[lab.item()]
    ax.text(x1, max(0, y1 - 4), f'{name} {sc:.2f}', color='white', fontsize=9,
            bbox=dict(facecolor='darkgreen', alpha=0.75, pad=1))
ax.set_title(f'COCO detections (Faster R-CNN ResNet-50-FPN) — {len(boxes)} boxes ≥ {SCORE_THR}', fontsize=12)
ax.axis('off')
plt.tight_layout()
plt.show()
print(f'Done: {len(boxes)} detections (COCO categories).')



Object detection localizes objects in images (bounding boxes + class labels). Modern detectors build upon powerful CNN backbones paired with **Feature Pyramid Networks (FPN)** for multi-scale detection.

```bash
Input Image
     │
  Backbone (CNN)                    ← Feature extraction
  C2, C3, C4, C5 (multi-scale)      ← Different resolutions
     │
  Neck (FPN / BiFPN / PANet)        ← Feature fusion
  P2, P3, P4, P5, P6               ← Enriched feature pyramid
     │
  Head (Detection / Segmentation)   ← Predictions
  Boxes, Scores, Masks
```

## 3.3 YOLO Series (v5 → YOLO26)

**YOLO (You Only Look Once)** is a single-stage detector — it predicts boxes and classes in a single
forward pass, making it extremely fast.

| Version | Year | Key Innovation |
|---|---|---|
| YOLOv5 | 2020 | CSP bottleneck, anchor-based, PyTorch-native tooling |
| YOLOv8 | 2023 | **C2f** backbone block, decoupled *anchor-free* head, one model for detect/segment/pose |
| YOLOv9 | 2024 | GELAN, Programmable Gradient Information (PGI) |
| YOLOv10 | 2024 | Consistent dual assignments → the first **NMS-free** YOLO |
| **YOLO11** | Sep 2024 | **C3k2** replaces C2f; adds a **C2PSA** attention block in the backbone; the mature default |
| **YOLO26** | Jan 2026 | Natively **end-to-end (no NMS)**; **DFL removed** from the head; up to 43% faster CPU ONNX inference than YOLO11n |

> ⚠️ **Two things to get right about this table.**
>
> 1. **"NMS-free" is not a property of the whole family.** It applies to **v10 and YOLO26 only**.
>    v5, v8 and v9 all still run Non-Maximum Suppression as a post-processing step. Any latency
>    argument that assumes "YOLO has no NMS" is wrong for v8/v9 and right for YOLO26.
> 2. **What we implement below is the *v8* block.** The C2f code in the next cell is YOLOv8's,
>    not the current generation's — we teach it because it shows the CSP idea most clearly. YOLO11
>    replaced it with C3k2 and bolted a C2PSA attention block on top.

**Sources:** [Ultralytics YOLO11](https://docs.ultralytics.com/models/yolo11/),
[Ultralytics YOLO26](https://docs.ultralytics.com/models/yolo26/).


### 3.3.1 Why YOLO “Only Looks Once”

**YOLO (You Only Look Once)** is named after its core idea: it performs **object detection in a single forward pass of the network**, instead of using multiple stages or repeated region proposals.



---

### 3.3.2 Traditional Object Detection (Before YOLO)

Earlier methods like:

* R-CNN
* Fast R-CNN
* Faster R-CNN

work in **multiple steps**:

1. Generate candidate regions (region proposals)
2. Run a CNN on each region
3. Classify and refine bounding boxes

This means the model effectively “looks” at the image **many times**, making it slower and more complex.



---

### 3.3.3 YOLO Approach (Single Pass)

YOLO reframes object detection as a **single regression problem**:

* Input: entire image
* Output: bounding boxes + class probabilities

The image is divided into a grid, and for each cell the model predicts:

* Bounding box coordinates
* Object confidence
* Class probabilities

This happens **all at once**, in one pass through the network.



---

### 3.3.4 Why “Only Once” Matters

- 1. Speed

* No region proposal stage
* Real-time detection (e.g., video, autonomous systems)

- 2. Global Context

* The model sees the **entire image simultaneously**
* Reduces false positives compared to patch-based methods

- 3. Simplicity

* End-to-end training
* Single unified architecture




### 3.3.5 YOLOv8 Architecture:
- **Backbone**: CSPDarknet with C2f blocks
- **Neck**: FPN + PAN (bi-directional feature fusion)
- **Head**: Anchor-free, decoupled (separate cls + reg branches)

*(This is the v8 layout, which the code below implements. YOLO11 keeps the anchor-free decoupled head but swaps C2f → **C3k2** and inserts a **C2PSA** attention block; YOLO26 additionally makes the head end-to-end so no NMS runs at all, and removes the DFL/`reg_max` distribution below.)*

<img src="https://raw.githubusercontent.com/EzequielMatiasArevalo/tuia-computer-vision/refs/heads/main/media/pictures/Class_2_Modern_CNN/yolo-anchor.png">





---

### 3.3.6 Summary

Instead of asking:

> “What objects exist in these many regions?”

YOLO asks:

> “What objects exist in this image, and where are they?”

And answers it **in one shot**.

YOLO “only looks once” because it:

* Processes the entire image **in a single forward pass**
* Predicts **all objects simultaneously**
* Eliminates the need for multi-stage pipelines

This design is what makes YOLO fast, efficient, and widely used in real-time computer vision systems.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# NOTE (see §3.3): C2f is the **YOLOv8** block, not the current generation's.
# YOLO11 replaced it with C3k2 and added a C2PSA attention block; YOLO26 is
# end-to-end (no NMS) and drops the DFL head that `reg_max` below implements.
# We implement C2f because it shows the CSP split/concat idea most clearly.
# ─────────────────────────────────────────────────────────────────────────────

class C2fBlock(nn.Module):
    """
    C2f (Cross-Stage Partial with 2 convolutions + feature flow) — YOLOv8.
    Inspired by CSP but with direct gradient flow paths.

    Splits channels, processes one part through N bottlenecks,
    then concatenates ALL intermediate outputs for rich feature reuse.
    """
    def __init__(self, in_ch: int, out_ch: int, n_bottlenecks: int = 1, shortcut: bool = True):
        super().__init__()
        hidden = out_ch // 2
        # Initial split conv
        self.cv1 = nn.Sequential(
            nn.Conv2d(in_ch, 2 * hidden, 1, bias=False),
            nn.BatchNorm2d(2 * hidden), nn.SiLU()
        )
        # Final aggregation conv
        self.cv2 = nn.Sequential(
            nn.Conv2d((2 + n_bottlenecks) * hidden, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch), nn.SiLU()
        )
        # Bottlenecks applied to the 2nd half
        self.bottlenecks = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(hidden, hidden, 3, padding=1, bias=False),
                nn.BatchNorm2d(hidden), nn.SiLU(),
                nn.Conv2d(hidden, hidden, 3, padding=1, bias=False),
                nn.BatchNorm2d(hidden), nn.SiLU(),
            ) for _ in range(n_bottlenecks)
        ])
        self.shortcut = shortcut and in_ch == out_ch

    def forward(self, x):
        # Split along channel dim
        out = list(self.cv1(x).chunk(2, dim=1))  # [left, right]
        # Process right part through bottlenecks, collect all intermediate feats
        for bn in self.bottlenecks:
            out.append(bn(out[-1]))
        # Concatenate all parts and fuse
        return self.cv2(torch.cat(out, dim=1))


class YOLOv8DetectionHead(nn.Module):
    """
    Simplified YOLOv8 Decoupled Detection Head.

    KEY CHANGE from YOLOv5: Decouples classification and regression.
    • Regression branch: predicts box offsets (4 values)
    • Classification branch: predicts class probabilities
    → Better gradient flow per task (no coupling)
    """
    def __init__(self, in_ch: int, num_classes: int = 80, reg_max: int = 16):
        super().__init__()
        hidden = max(in_ch, 64)
        # Regression branch (4 box coords × reg_max)
        self.reg_branch = nn.Sequential(
            nn.Conv2d(in_ch, hidden, 3, padding=1), nn.SiLU(),
            nn.Conv2d(hidden, hidden, 3, padding=1), nn.SiLU(),
            nn.Conv2d(hidden, 4 * reg_max, 1)
        )
        # Classification branch
        self.cls_branch = nn.Sequential(
            nn.Conv2d(in_ch, hidden, 3, padding=1), nn.SiLU(),
            nn.Conv2d(hidden, hidden, 3, padding=1), nn.SiLU(),
            nn.Conv2d(hidden, num_classes, 1)
        )

    def forward(self, feat):
        box  = self.reg_branch(feat)   # (B, 4*reg_max, H, W)
        cls  = self.cls_branch(feat)   # (B, num_classes, H, W)
        return box, cls


# ── Test C2f and detection head ───────────────────────────────────────────────
x   = torch.randn(2, 256, 40, 40)   # P4 feature map
c2f = C2fBlock(256, 256, n_bottlenecks=3)
det = YOLOv8DetectionHead(256, num_classes=80)

feat     = c2f(x)
box, cls = det(feat)

print('🎯 YOLOv8 Building Blocks')
print(f'  Input feat    : {x.shape}')
print(f'  C2f output    : {feat.shape}')
print(f'  Box output    : {box.shape}  (4 × reg_max per cell)')
print(f'  Class output  : {cls.shape}  (80 COCO classes per cell)')
print(f'  C2f params    : {count_params(c2f):,}')
print(f'  Head params   : {count_params(det):,}')

---
## 3.4 Mask R-CNN

The summary table credits Mask R-CNN with RoIAlign, and the TOC promises the section — here it is.

**Mask R-CNN** ([He et al., 2017](https://arxiv.org/abs/1703.06870)) is the smallest possible step
from detection to **instance segmentation**: take Faster R-CNN from §3.2 and add a **third branch**
to the RoI head that predicts a binary mask for each region. Nothing else about the pipeline changes.

```text
                          ┌── class scores  (K + 1)
RoI feature ─── RoIAlign ─┼── box deltas    (4 per class)
   (from FPN)             └── mask branch   (small FCN → K masks of m × m)     ← new
```

#### The three ideas that make it work

**1. RoIAlign instead of RoI Pooling.** This is the paper's headline contribution and it is
explained in §3.2.2 above. Box classification tolerates half-cell rounding; a pixel-level mask does
not, and removing the quantization is what lifted mask quality enough to make the branch useful.

**2. The mask branch is a tiny FCN, not fully connected.** It is a handful of convolutions applied
to the RoI feature, producing an `m × m` (typically 28×28) mask that is then resized to the box.
Keeping it convolutional preserves the spatial layout that an FC layer would flatten away, and costs
very few parameters.

**3. Mask and class prediction are decoupled.** The branch outputs **K** masks per RoI — one per
class — and the loss only looks at the channel of the *ground-truth* class, applying a per-pixel
**sigmoid + binary cross-entropy** to it. There is no softmax competition between classes inside a
pixel. This matters: the alternative (one softmax over classes per pixel) makes the mask branch
compete with the classification branch and measurably hurts both.

$$\mathcal{L} = \mathcal{L}_{cls} + \mathcal{L}_{box} + \mathcal{L}_{mask}$$

#### In PyTorch

```python
from torchvision.models.detection import maskrcnn_resnet50_fpn, MaskRCNN_ResNet50_FPN_Weights

weights = MaskRCNN_ResNet50_FPN_Weights.DEFAULT
model = maskrcnn_resnet50_fpn(weights=weights).eval()

out = model([img_tensor])[0]
out['masks']   # (N, 1, H, W) soft masks in [0, 1] — threshold at ~0.5
out['boxes']   # (N, 4)   out['labels'] (N,)   out['scores'] (N,)
```

The one gotcha compared with the Faster R-CNN cell above: `masks` come back as **soft** probability
maps at full image resolution, one per detection, so you threshold them yourself.

#### Where it sits today

Mask R-CNN is a two-stage method, so it is accurate and slow — it remains the reference baseline for
instance segmentation and the standard starting point when you fine-tune on a small custom dataset.
For real-time work the single-stage equivalents (`yolo11-seg`, `yolo26-seg`, which predict mask
*coefficients* combined with a set of learned prototypes rather than per-RoI masks) are what you
would actually deploy — but they all inherit the RoIAlign-style "don't quantize your coordinates"
lesson.

---
## 3.5 EfficientDet

**EfficientDet** (Tan et al., 2020 — Google) applies the **compound scaling** principle from EfficientNet to object detection. It introduces **BiFPN (Bidirectional FPN)** — a learnable, weighted feature pyramid that fuses information in both directions.

#### BiFPN vs Classic FPN:
```bash
FPN (top-down only):      P6 → P5 → P4 → P3
PANet (+ bottom-up):      P3 → P4 → P5 → P6
BiFPN (bidirectional):    Both, with learnable weights + skip connections
```

**Weighted fusion**: Instead of simple sum/concatenation, BiFPN learns per-feature weights:
$$P^{td}_{i} = \frac{w_1 P_i + w_2 \cdot resize(P_{i+1})}{w_1 + w_2 + \epsilon}$$

In [ ]:
class BiFPNLayer(nn.Module):
    """
    BiFPN Layer — Bidirectional Feature Pyramid Network.

    Fuses features from P3 to P7 in both top-down and bottom-up directions
    with LEARNABLE weights (fast normalized fusion).
    """
    def __init__(self, num_channels: int = 64, num_levels: int = 5, eps: float = 1e-4):
        super().__init__()
        self.num_levels = num_levels
        self.eps = eps
        # Depthwise separable convolutions for feature refinement
        self.convs = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(num_channels, num_channels, 3, padding=1, groups=num_channels, bias=False),
                nn.Conv2d(num_channels, num_channels, 1, bias=False),
                nn.BatchNorm2d(num_channels),
                nn.SiLU()
            ) for _ in range(num_levels * 2 - 2)  # top-down + bottom-up paths
        ])
        # Learnable fusion weights (initialized to 1 = equal weight)
        self.td_weights = nn.Parameter(torch.ones(num_levels - 1, 2))  # top-down
        self.bu_weights = nn.Parameter(torch.ones(num_levels - 1, 3))  # bottom-up

    def weighted_sum(self, feats, weights):
        """Fast normalized weighted fusion."""
        w = F.relu(weights)  # ensure positive
        w = w / (w.sum() + self.eps)
        return sum(w[i] * feats[i] for i in range(len(feats)))

    def forward(self, features: list) -> list:
        # features: [P3, P4, P5, P6, P7]
        # ── TOP-DOWN PASS (high-level → low-level) ─────────────
        td_features = [features[-1]]  # Start from P7
        for i in range(self.num_levels - 2, -1, -1):
            upsampled = F.interpolate(td_features[-1], size=features[i].shape[2:], mode='nearest')
            fused = self.weighted_sum([features[i], upsampled], self.td_weights[self.num_levels - 2 - i])
            td_features.append(self.convs[self.num_levels - 2 - i](fused))
        td_features = td_features[::-1]  # reverse to [P3_td, P4_td, ..., P7_td]
        # ── BOTTOM-UP PASS (low-level → high-level) ─────────────
        out_features = [td_features[0]]  # Start from P3_td
        for i in range(1, self.num_levels):
            downsampled = F.max_pool2d(out_features[-1], kernel_size=2)
            if i < self.num_levels - 1:
                fused = self.weighted_sum(
                    [features[i], td_features[i], downsampled],
                    self.bu_weights[i - 1]
                )
            else:
                fused = self.weighted_sum([features[i], downsampled], self.bu_weights[i-1][:2])
            out_features.append(self.convs[self.num_levels - 1 + i - 1](fused))
        return out_features


# ── Test BiFPN with dummy FPN features ───────────────────────────────────────
C = 64   # BiFPN channels
# Simulate FPN outputs [P3, P4, P5, P6, P7] at decreasing resolutions
feats = [
    torch.randn(2, C, 64, 64),  # P3
    torch.randn(2, C, 32, 32),  # P4
    torch.randn(2, C, 16, 16),  # P5
    torch.randn(2, C, 8,  8 ),  # P6
    torch.randn(2, C, 4,  4 ),  # P7
]

bifpn = BiFPNLayer(num_channels=C, num_levels=5)
out   = bifpn(feats)

print('🔀 BiFPN — Bidirectional Feature Pyramid')
print(f'  BiFPN params: {count_params(bifpn):,}')
print('\n  Input (FPN features)  →  Output (BiFPN features):')
levels = ['P3', 'P4', 'P5', 'P6', 'P7']
for i, (inp, oup, lvl) in enumerate(zip(feats, out, levels)):
    print(f'    {lvl}: {tuple(inp.shape[2:])} → {tuple(oup.shape[2:])}  (same spatial, richer semantics)')

---
<a name='4-key-trends'></a>
# 4. Key Trends and Characteristics





## 4.1 Attention Mechanisms in CNNs

### SE and CBAM — Channel & Spatial Attention Modules

Both modules answer the same fundamental question: not all features are equally useful — how do we let the network learn which ones to emphasize? They're lightweight add-ons that slot into any CNN architecture and recalibrate feature maps before passing them forward.

<img src="https://raw.githubusercontent.com/EzequielMatiasArevalo/tuia-computer-vision/refs/heads/main/media/pictures/Class_2_Modern_CNN/SE-2.png">


Modern CNNs incorporate **channel attention** and **spatial attention** to focus on the most relevant features and locations:

### Squeeze-and-Excitation (SE) — Channel Attention
- **Squeeze**: Global average pool → compress spatial info into a vector
- **Excite**: Two FC layers learn channel-wise importance weights
- **Scale**: Multiply feature maps by these weights

<img src="https://raw.githubusercontent.com/EzequielMatiasArevalo/tuia-computer-vision/refs/heads/main/media/pictures/Class_2_Modern_CNN/SE.png" widht=400 heigth=400>

### CBAM — Convolutional Block Attention Module

Applies **channel attention THEN spatial attention** sequentially.

<p>
CBAM goes further: two sequential attention modules, one for channels and one for spatial locations. It asks both which channels matter and where to look.

</p>

<img src="https://raw.githubusercontent.com/EzequielMatiasArevalo/tuia-computer-vision/refs/heads/main/media/pictures/Class_2_Modern_CNN/cbam.png" widht=400 heigth=400>

<img src="https://raw.githubusercontent.com/EzequielMatiasArevalo/tuia-computer-vision/refs/heads/main/media/pictures/Class_2_Modern_CNN/cbam-2.png" widht=400 heigth=400>

### Intuition in one sentence each

* SE — "Look at what each detector found across the whole image, then decide which detectors to trust for this particular input."
> SE squeezes each channel to a scalar via Global Average Pool, then excites through two FC layers. Output: C scalar weights multiplied back onto each channel. Spatial structure is ignored entirely.

<img src="https://raw.githubusercontent.com/EzequielMatiasArevalo/tuia-computer-vision/refs/heads/main/media/pictures/Class_2_Modern_CNN/SE-Output.png" widht=400 heigth=400>

* CBAM — "First decide which detectors to trust, then decide which regions of the image to pay attention to — then pass forward only what matters both in feature type and location."

> Spatial attention collapses channels into 2 maps (avg and max), then uses a 7×7 conv to predict which spatial locations matter. High values = important regions. The network learns to focus on the object, not the background.

<img src="https://raw.githubusercontent.com/EzequielMatiasArevalo/tuia-computer-vision/refs/heads/main/media/pictures/Class_2_Modern_CNN/cbam-Output.png" widht=400 heigth=400>

In [ ]:
class SEBlock(nn.Module):
    """
    Squeeze-and-Excitation Block (Hu et al., 2018).
    Recalibrates channel-wise feature responses adaptively.

    ratio: reduction ratio (higher = fewer params in excitation)
    """
    def __init__(self, channels: int, ratio: int = 16):
        super().__init__()
        self.squeeze  = nn.AdaptiveAvgPool2d(1)   # Global average pooling
        self.excite   = nn.Sequential(
            nn.Flatten(),
            nn.Linear(channels, channels // ratio, bias=False),  # compress
            nn.ReLU(inplace=True),
            nn.Linear(channels // ratio, channels, bias=False),  # expand
            nn.Sigmoid()
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, C, H, W = x.shape
        # Squeeze: (B,C,H,W) → (B,C,1,1) → (B,C)
        s = self.squeeze(x)
        # Excite: (B,C) → (B,C) channel weights in [0,1]
        e = self.excite(s).view(B, C, 1, 1)
        # Scale: channel-wise recalibration
        return x * e


class CBAM(nn.Module):
    """
    CBAM — Convolutional Block Attention Module (Woo et al., 2018).
    Applies sequential channel + spatial attention.
    """
    def __init__(self, channels: int, ratio: int = 16, kernel_size: int = 7):
        super().__init__()
        # --- Channel Attention ---
        self.avg_pool  = nn.AdaptiveAvgPool2d(1)
        self.max_pool  = nn.AdaptiveMaxPool2d(1)
        self.chan_mlp  = nn.Sequential(
            nn.Flatten(),
            nn.Linear(channels, channels // ratio, bias=False), nn.ReLU(),
            nn.Linear(channels // ratio, channels, bias=False)
        )
        # --- Spatial Attention ---
        padding = (kernel_size - 1) // 2
        self.spat_conv = nn.Conv2d(2, 1, kernel_size, padding=padding, bias=False)

    def channel_attention(self, x):
        B, C, _, _ = x.shape
        avg = self.chan_mlp(self.avg_pool(x))
        mx  = self.chan_mlp(self.max_pool(x))
        return torch.sigmoid(avg + mx).view(B, C, 1, 1)

    def spatial_attention(self, x):
        # Compute avg and max along channel dim → 2-channel map
        avg = x.mean(dim=1, keepdim=True)
        mx  = x.max(dim=1, keepdim=True).values
        combined = torch.cat([avg, mx], dim=1)  # (B, 2, H, W)
        return torch.sigmoid(self.spat_conv(combined))

    def forward(self, x):
        x = x * self.channel_attention(x)   # Channel attention
        x = x * self.spatial_attention(x)   # Spatial attention
        return x


# ── Visualize attention maps ──────────────────────────────────────────────────
torch.manual_seed(0)
x   = torch.randn(1, 64, 32, 32)
se  = SEBlock(64, ratio=16)
cbam = CBAM(64)

with torch.no_grad():
    # SE channel weights
    se_weights = se.excite(se.squeeze(x).flatten(1)).squeeze().numpy()
    # CBAM spatial map
    x_chan = x * cbam.channel_attention(x)
    spat_map = cbam.spatial_attention(x_chan).squeeze().numpy()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# SE channel weights
axes[0].bar(range(len(se_weights)), se_weights, color=plt.cm.RdYlGn(se_weights))
axes[0].axhline(0.5, color='red', linestyle='--', alpha=0.6, label='threshold=0.5')
axes[0].set_title('SE Block — Channel Attention Weights\n(each value = importance of one channel)', fontsize=10)
axes[0].set_xlabel('Channel Index'); axes[0].set_ylabel('Attention Weight')
axes[0].legend()

# CBAM spatial map
im = axes[1].imshow(spat_map, cmap='hot', aspect='auto')
plt.colorbar(im, ax=axes[1])
axes[1].set_title('CBAM — Spatial Attention Map\n(bright = attended regions)', fontsize=10)
axes[1].set_xlabel('Width'); axes[1].set_ylabel('Height')

# Parameter comparison
models_info = [
    ('No Attention', 0),
    ('SE Block (r=16)', count_params(se)),
    ('CBAM (r=16)', count_params(cbam)),
]
names, params = zip(*models_info)
bars = axes[2].barh(names, params, color=['#aaaaaa', '#4CAF50', '#2196F3'])
axes[2].set_title('Attention Module Overhead\n(extra params for 64 channels)', fontsize=10)
axes[2].set_xlabel('Additional Parameters')
for bar, p in zip(bars, params):
    axes[2].text(p + 5, bar.get_y() + bar.get_height()/2, f'{p:,}', va='center', fontsize=9)

plt.suptitle('Attention Mechanisms in Modern CNNs', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('attention_mechanisms.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'SE Block params   : {count_params(se):,}')
print(f'CBAM params       : {count_params(cbam):,}')
print('→ Very small overhead for significant accuracy gains!')

---
## 4.2 Efficiency vs Performance: FLOPs Analysis

A central challenge in modern CV: **maximize accuracy per compute budget (FLOPs)**.

The **FLOPs** (Floating Point Operations) of a conv layer are:
$$\text{FLOPs} = 2 \times C_{in} \times C_{out} \times k^2 \times H \times W$$

Key efficiency techniques:
- **Depthwise Separable Conv**: reduces FLOPs by ~8-9×
- **Grouped Convolutions**: reduces FLOPs by `1/groups`
- **Pruning**: removes low-magnitude weights
- **Knowledge Distillation**: small student learns from large teacher

In [ ]:
def conv_flops(Cin, Cout, k, H, W, groups=1):
    """FLOPs for a convolution layer (multiply-accumulate × 2)."""
    return 2 * Cin * Cout * (k ** 2) * H * W / groups

def depthwise_sep_flops(Cin, Cout, k, H, W):
    """Depthwise separable = DWConv + PWConv."""
    dw = conv_flops(Cin, Cin, k, H, W, groups=Cin)   # DWConv (groups=Cin)
    pw = conv_flops(Cin, Cout, 1, H, W)               # 1×1 PWConv
    return dw + pw

# ── FLOPs comparison for a typical layer ─────────────────────────────────────
Cin, Cout, k, H, W = 128, 256, 3, 56, 56

std_flops  = conv_flops(Cin, Cout, k, H, W)
dws_flops  = depthwise_sep_flops(Cin, Cout, k, H, W)
g4_flops   = conv_flops(Cin, Cout, k, H, W, groups=4)
g8_flops   = conv_flops(Cin, Cout, k, H, W, groups=8)

results = [
    ('Standard Conv 3×3', std_flops, 'baseline'),
    ('Grouped Conv (g=4)', g4_flops, f'{g4_flops/std_flops:.2%}'),
    ('Grouped Conv (g=8)', g8_flops, f'{g8_flops/std_flops:.2%}'),
    ('Depthwise Sep.', dws_flops, f'{dws_flops/std_flops:.2%}'),
]

print(f'📊 FLOPs Comparison  (Cin={Cin}, Cout={Cout}, 3×3, H=W={H})')
print(f'  {"Method":<25} {"GFLOPs":>10}  {"vs. Standard"}')
print('  ' + '-'*50)
for name, flops, ratio in results:
    print(f'  {name:<25} {flops/1e9:>10.2f}  {ratio}')

# Visualization
fig, ax = plt.subplots(figsize=(10, 4))
names_plot = [r[0] for r in results]
gflops     = [r[1]/1e9 for r in results]
colors     = ['#e74c3c', '#e67e22', '#f1c40f', '#2ecc71']
bars = ax.barh(names_plot, gflops, color=colors, edgecolor='k', linewidth=0.8)
for bar, g in zip(bars, gflops):
    ax.text(g + 0.05, bar.get_y() + bar.get_height()/2,
            f'{g:.2f} G', va='center', fontsize=10, fontweight='bold')
ax.set_xlabel('GFLOPs (lower = more efficient)', fontsize=11)
ax.set_title('Computational Cost by Convolution Type', fontsize=13, fontweight='bold')
ax.axvline(std_flops/1e9, color='red', linestyle='--', alpha=0.4, label='Standard baseline')
ax.legend()
plt.tight_layout()
#plt.savefig('figures/flops_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 4.3 1D-CNNs for Temporal Signal Analysis

**1D-CNNs** apply convolutions along the **time dimension** instead of spatial dimensions. They are particularly effective for:
- 📈 Time-series classification (ECG, EEG, accelerometer)
- 🎵 Audio signal processing
- 🧬 Genomic sequence analysis

The same architectural improvements (residual connections, attention, depthwise conv) apply to 1D.

In [ ]:
class Residual1DBlock(nn.Module):
    """
    1D Residual Block for temporal signal classification.
    Architecture identical to ResNet but with nn.Conv1d.

    Input:  (Batch, Channels, Time)
    Output: (Batch, Channels, Time)
    """
    def __init__(self, in_ch: int, out_ch: int, kernel_size: int = 7, stride: int = 1):
        super().__init__()
        padding = kernel_size // 2
        self.block = nn.Sequential(
            nn.Conv1d(in_ch, out_ch, kernel_size, stride=stride, padding=padding, bias=False),
            nn.BatchNorm1d(out_ch), nn.ReLU(inplace=True),
            nn.Conv1d(out_ch, out_ch, kernel_size, padding=padding, bias=False),
            nn.BatchNorm1d(out_ch)
        )
        self.shortcut = nn.Sequential(
            nn.Conv1d(in_ch, out_ch, 1, stride=stride, bias=False),
            nn.BatchNorm1d(out_ch)
        ) if in_ch != out_ch or stride != 1 else nn.Identity()
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.relu(self.block(x) + self.shortcut(x))


class TCNClassifier(nn.Module):
    """
    Temporal CNN Classifier — designed for ECG / time-series tasks.
    Uses dilated convolutions to capture long-range temporal dependencies.
    """
    def __init__(self, in_channels: int = 1, num_classes: int = 5):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(in_channels, 32, kernel_size=15, padding=7, bias=False),
            nn.BatchNorm1d(32), nn.ReLU(inplace=True)
        )
        self.stages = nn.Sequential(
            Residual1DBlock(32, 64,  stride=2),
            Residual1DBlock(64, 128, stride=2),
            Residual1DBlock(128, 256, stride=2),
        )
        # Channel attention (1D SE block)
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(256, 16), nn.ReLU(),
            nn.Linear(16, 256), nn.Sigmoid()
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.stages(x)
        # Apply 1D channel attention
        w = self.se(x).unsqueeze(-1)
        x = x * w
        return self.classifier(x)


# ── ECG signal simulation ─────────────────────────────────────────────────────
t     = np.linspace(0, 4, 1000)  # 4 seconds at 250Hz
ecg   = (np.sin(2 * np.pi * 1.2 * t) + 0.3 * np.sin(2 * np.pi * 5 * t)
         + 0.1 * np.random.randn(len(t)))

fig, axes = plt.subplots(2, 1, figsize=(12, 5))
axes[0].plot(t, ecg, color='crimson', linewidth=0.8)
axes[0].set_title('Simulated ECG Signal', fontsize=11)
axes[0].set_xlabel('Time (s)'); axes[0].set_ylabel('Amplitude')
axes[0].grid(True, alpha=0.3)

# Show how 1D conv kernel slides over time
kernel_size = 15
demo_filter = np.hamming(kernel_size)
feature = np.convolve(ecg, demo_filter, mode='same')
axes[1].plot(t, feature, color='navy', linewidth=0.8, label='Conv1D output (smoothed)')
axes[1].set_title('Feature map after 1D Convolution (kernel_size=15)', fontsize=11)
axes[1].set_xlabel('Time (s)'); axes[1].set_ylabel('Feature Response')
axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout()
#plt.savefig('ecg_1dcnn.png', dpi=120, bbox_inches='tight')
plt.show()

# Test the 1D CNN
model_1d = TCNClassifier(in_channels=1, num_classes=5)
ecg_batch = torch.tensor(ecg, dtype=torch.float32).unsqueeze(0).unsqueeze(0)  # (1,1,1000)
with torch.no_grad():
    logits = model_1d(ecg_batch)

print('🫀 1D CNN for ECG Classification')
print(f'  Input  : (batch=1, channels=1, timesteps=1000)')
print(f'  Output : {logits.shape}  (5 heart rhythm classes)')
print(f'  Params : {count_params(model_1d)/1e3:.1f}K')

---
<a name='5-transfer-learning'></a>
# 5. Transfer Learning: Fine-Tuning a Pretrained CNN

**Transfer Learning** is the single most impactful practice in modern CV. Instead of training from scratch:

1. Start from a model pretrained on **ImageNet** (1.2M images, 1000 classes)
2. **Replace** the classification head for your task
3. **Fine-tune** all or part of the network



## 5.1 Fine-tuning strategies
| Strategy | Frozen layers | When to use |
|---|---|---|
| **Feature extraction** | All backbone | Very small dataset (<1K images) |
| **Partial fine-tuning** | Early stages | Small-medium dataset |
| **Full fine-tuning** | None | Large dataset, similar domain |
| **Progressive unfreezing** | Layer by layer | Best results, prevents catastrophic forgetting |

In [ ]:
class TransferLearningModel(nn.Module):
    """
    Demonstrates three fine-tuning strategies:
    1. Feature extraction   — backbone frozen, only head trained
    2. Partial fine-tuning  — last stage + head trained
    3. Full fine-tuning     — entire network trained (lower LR)
    """
    def __init__(self, backbone_name: str, num_classes: int, strategy: str = 'partial'):
        super().__init__()
        # Load pretrained backbone
        self.backbone = timm.create_model(
            backbone_name,
            pretrained=True,       # ← the whole point. With pretrained=False there is no
                                   #   transfer at all: you are training from random init.
            num_classes=0,         # Remove original head
            global_pool='avg'      # Global average pool
        )
        feat_dim = self.backbone.num_features
        # Custom head for new task
        self.head = nn.Sequential(
            nn.LayerNorm(feat_dim),
            nn.Dropout(p=0.2),
            nn.Linear(feat_dim, 256),
            nn.GELU(),
            nn.Linear(256, num_classes)
        )
        # Apply strategy
        self._apply_strategy(strategy)

    def _apply_strategy(self, strategy: str):
        if strategy == 'feature_extraction':
            # Freeze entire backbone
            for p in self.backbone.parameters():
                p.requires_grad = False

        elif strategy == 'partial':
            # Freeze all backbone, then unfreeze last stage
            for p in self.backbone.parameters():
                p.requires_grad = False
            # Unfreeze last ConvNeXt stage
            for p in list(self.backbone.parameters())[-30:]:
                p.requires_grad = True

        elif strategy == 'full':
            # All parameters trainable (use lower LR for backbone)
            for p in self.backbone.parameters():
                p.requires_grad = True

    def get_optimizer_groups(self, head_lr: float = 1e-3, backbone_lr: float = 1e-5):
        """Differential learning rates: head gets higher LR than backbone."""
        return [
            {'params': self.backbone.parameters(), 'lr': backbone_lr},
            {'params': self.head.parameters(),     'lr': head_lr},
        ]

    def forward(self, x):
        features = self.backbone(x)  # (B, feat_dim)
        return self.head(features)


# ── Compare the *parameter budget* of each strategy ──────────────────────────
# This cell only counts parameters. Section 5.2 below actually trains them and adds
# validation accuracy, which is the column that decides between the three.
print('📦 Transfer Learning Strategies — ConvNeXt-Tiny → CIFAR-10 (10 classes) — parameter budget')
print(f'  {"Strategy":<25} {"Trainable":>12}  {"Frozen":>10}')
print('  ' + '-'*55)

for strategy in ['feature_extraction', 'partial', 'full']:
    model_tl = TransferLearningModel('convnext_tiny', num_classes=10, strategy=strategy)
    trainable = sum(p.numel() for p in model_tl.parameters() if p.requires_grad)
    frozen    = sum(p.numel() for p in model_tl.parameters() if not p.requires_grad)
    print(f'  {strategy:<25} {trainable/1e6:>10.2f}M  {frozen/1e6:>8.2f}M')

print('\n  Recommendation:')
print('  • Small dataset   → feature_extraction (avoid overfitting)')
print('  • Medium dataset  → partial (good balance)')
print('  • Large dataset   → full (with differential LR)')

In [ ]:
# ── Progressive unfreezing scheduler ─────────────────────────────────────────
def progressive_unfreeze(
    model: TransferLearningModel,
    epoch: int,
    unfreeze_schedule: dict
):
    """
    Progressively unfreezes backbone layers epoch by epoch.
    This avoids catastrophic forgetting of pretrained features.

    unfreeze_schedule: {epoch: num_params_to_unfreeze_from_end}
    """
    if epoch in unfreeze_schedule:
        n = unfreeze_schedule[epoch]
        params = list(model.backbone.parameters())
        # Unfreeze last n parameters
        for p in params[-n:]:
            p.requires_grad = True
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f'  Epoch {epoch:>2}: Unfreezing {n} param-groups → {trainable/1e6:.2f}M trainable')


# Simulate 10 epochs of progressive unfreezing
model_prog = TransferLearningModel('convnext_tiny', num_classes=10, strategy='feature_extraction')
schedule = {0: 0, 2: 20, 4: 60, 6: 120, 8: 999}  # epoch: unfreeze N last groups

print('🔓 Progressive Unfreezing Schedule:')
for ep in range(0, 10, 2):
    progressive_unfreeze(model_prog, ep, schedule)

## 5.2 Hands on — actually fine-tuning ConvNeXt-Tiny on CIFAR-10

Section 5.1 counted parameters and printed an unfreezing schedule. Neither of those tells you
whether freezing the backbone **costs accuracy**, which is the only question the three strategies
exist to answer. This section closes the gap: it downloads CIFAR-10, preprocesses it the way the
pretrained backbone expects, runs a short training loop per strategy, and reports **validation
accuracy** as the third column.

Two details matter more than whatever numbers you get:

- **`pretrained=True`.** The previous cell used to build the backbone with `pretrained=False`. That
  is not transfer learning — it is ConvNeXt-Tiny from random initialization wearing the label.
- **Resize to 224² and normalize with the pretraining statistics.** CIFAR-10 is 32×32. ConvNeXt's
  stem downsamples by 4 and each of its four stages by 2 more, so a 32×32 input arrives at the head
  as a 1×1 feature map. The transform is part of the pretrained model, not an afterthought — the
  same mistake shows up in the remote-sensing lab of this course.

Budget a few minutes per strategy on a T4. The subset sizes and `EPOCHS` are at the top of the cell:
lower them if you are short on time. **The ordering of the three strategies is the lesson, not the
absolute accuracies** — one epoch on 5,000 images is nowhere near convergence.

In [ ]:
# ── 5.2 Fine-tune for real: ConvNeXt-Tiny → CIFAR-10 ─────────────────────────
import time
from torch.utils.data import DataLoader, Subset
from torchvision.datasets import CIFAR10

N_TRAIN, N_VAL, EPOCHS, BS = 5000, 2000, 1, 32      # shrink these if you are short on time

# The backbone was pretrained on ImageNet at 224² with ImageNet statistics —
# feed it anything else and you are measuring a preprocessing mismatch.
IMAGENET_MEAN, IMAGENET_STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
cifar_tf = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_full = CIFAR10('./data', train=True,  download=True, transform=cifar_tf)
val_full   = CIFAR10('./data', train=False, download=True, transform=cifar_tf)

g = torch.Generator().manual_seed(0)
train_ds = Subset(train_full, torch.randperm(len(train_full), generator=g)[:N_TRAIN].tolist())
val_ds   = Subset(val_full,   torch.randperm(len(val_full),   generator=g)[:N_VAL].tolist())

train_loader = DataLoader(train_ds, batch_size=BS, shuffle=True,  num_workers=2, pin_memory=True)
val_loader_c = DataLoader(val_ds,   batch_size=64, shuffle=False, num_workers=2, pin_memory=True)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    correct = total = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        correct += (model(xb).argmax(dim=1) == yb).sum().item()
        total   += yb.numel()
    return 100.0 * correct / total


def run_strategy(strategy: str, epochs: int = EPOCHS):
    torch.manual_seed(0)
    model = TransferLearningModel('convnext_tiny', num_classes=10, strategy=strategy).to(device)
    optimizer = torch.optim.AdamW(
        model.get_optimizer_groups(head_lr=1e-3, backbone_lr=1e-5), weight_decay=1e-4
    )
    criterion = nn.CrossEntropyLoss()

    t0 = time.perf_counter()
    for _ in range(epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad(set_to_none=True)
            criterion(model(xb), yb).backward()
            optimizer.step()
    elapsed = time.perf_counter() - t0

    acc = evaluate(model, val_loader_c)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return trainable, acc, elapsed


print(f'🎯 ConvNeXt-Tiny → CIFAR-10  ({N_TRAIN} train / {N_VAL} val images, {EPOCHS} epoch(s))')
print(f'  {"Strategy":<22}{"Trainable":>12}{"Val acc":>10}{"Train s":>10}')
print('  ' + '-' * 54)
for strategy in ['feature_extraction', 'partial', 'full']:
    n_params, accuracy, secs = run_strategy(strategy)
    print(f'  {strategy:<22}{n_params/1e6:>10.2f}M{accuracy:>9.2f}%{secs:>10.1f}')

print('\n  Read the table, not the theory: if `feature_extraction` is already close to `full`,')
print('  your dataset is small and close to ImageNet — freeze and save the compute.')

In [ ]:
# ── Final Summary: Architecture Comparison Dashboard ─────────────────────────
arch_data = {
    'Architecture': ['ResNet-50', 'ResNeXt-50 (32×4d)', 'EfficientNetV2-S',
                     'ConvNeXt-Tiny', 'NFNet-F0', 'WideResNet-50-2'],
    'Params (M)':   [25.6, 25.0, 21.5, 28.6, 71.5, 68.9],
    'GFLOPs':       [4.1, 4.3, 8.8, 4.5, 12.4, 11.4],
    'Top-1 Acc':    [76.1, 77.6, 83.9, 82.1, 83.6, 81.8],
    'BatchNorm':    ['Yes', 'Yes', 'Yes', 'Yes', 'NO', 'Yes'],
    'Attention':    ['No', 'No', 'SE', 'No', 'No', 'No'],
    'Year':         [2015, 2017, 2021, 2022, 2021, 2016]
}

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = ['#E74C3C','#E67E22','#F39C12','#2ECC71','#3498DB','#9B59B6']
names = arch_data['Architecture']

# Plot 1: Params vs Accuracy
for i, (p, a, n) in enumerate(zip(arch_data['Params (M)'], arch_data['Top-1 Acc'], names)):
    axes[0].scatter(p, a, s=150, color=colors[i], zorder=5, edgecolors='k', lw=1.5)
    axes[0].annotate(n.split()[0], (p, a), textcoords='offset points', xytext=(5, 3), fontsize=7)
axes[0].set_xlabel('Parameters (M)', fontsize=10)
axes[0].set_ylabel('Top-1 Accuracy (%)', fontsize=10)
axes[0].set_title('Params vs Accuracy', fontsize=11, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Plot 2: FLOPs vs Accuracy
for i, (f, a, n) in enumerate(zip(arch_data['GFLOPs'], arch_data['Top-1 Acc'], names)):
    axes[1].scatter(f, a, s=150, color=colors[i], zorder=5, edgecolors='k', lw=1.5)
    axes[1].annotate(n.split()[0], (f, a), textcoords='offset points', xytext=(5, 3), fontsize=7)
axes[1].set_xlabel('GFLOPs', fontsize=10)
axes[1].set_ylabel('Top-1 Accuracy (%)', fontsize=10)
axes[1].set_title('FLOPs vs Accuracy', fontsize=11, fontweight='bold')
axes[1].grid(True, alpha=0.3)

# Plot 3: Accuracy bar chart
short_names = [n.split()[0] for n in names]
bars = axes[2].barh(short_names, arch_data['Top-1 Acc'],
                    color=colors, edgecolor='k', linewidth=0.8)
axes[2].set_xlabel('ImageNet Top-1 Accuracy (%)', fontsize=10)
axes[2].set_title('ImageNet Accuracy Comparison', fontsize=11, fontweight='bold')
axes[2].set_xlim(74, 85)
for bar, acc in zip(bars, arch_data['Top-1 Acc']):
    axes[2].text(acc + 0.05, bar.get_y() + bar.get_height()/2,
                 f'{acc}%', va='center', fontsize=9, fontweight='bold')

plt.suptitle('Modern CNN Architectures — ImageNet Benchmark Comparison',
             fontsize=14, fontweight='bold')
plt.tight_layout()
#plt.savefig('architecture_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

> ### ⚠️ How to read this chart: it mixes **training recipes**, not just architectures
>
> Every Top-1 number above is taken from the model's own paper, and those papers used very different
> training setups. ResNet-50's **76.1%** is the 2015 recipe (90 epochs, SGD with a step schedule,
> basic crop-and-flip augmentation). ConvNeXt-Tiny's **82.1%** comes from a 300-epoch AdamW recipe
> with Mixup, CutMix, RandAugment, stochastic depth and label smoothing. EfficientNetV2-S's **83.9%**
> is measured at **384px**, not at the 224px the others use.
>
> So how much of the gap is really the *architecture*? The ConvNeXt paper answers this directly, and
> it is the most useful single fact on this page: a **plain, unmodified ResNet-50 trained with the
> modern recipe goes from 76.1% → 78.8%**. That is roughly **45% of the entire ResNet-50 →
> ConvNeXt-Tiny gap**, obtained before changing a single layer.
>
> The chart is still worth having — the params/FLOPs axes are comparable and the ordering is broadly
> right — but "architecture X beats architecture Y by N points" is not a claim you can read off it.
>
> **Exercise.** Add a second ResNet-50 entry at 78.8% (label it `ResNet-50 (modern recipe)`) to
> `arch_data` and re-run the cell. Which of the architectural conclusions you would have drawn from
> the original chart survive?

---
<a name='6-summary'></a>
# 📚 Summary

| Topic | Key Takeaway |
|---|---|
| **Normalization** | BN averages over (N,H,W) per channel; LN over (C,H,W) per sample; GN over (C/G,H,W) per group. Only BN uses the batch. |
| **Why BN works** | Motivated by internal covariate shift (2015), but explained today by **loss-landscape smoothing** (Santurkar et al., 2018) |
| **Activations** | ReLU → GELU / SiLU: smooth, self-gating, and free. ConvNeXt also uses *one* activation per block, not one per conv |
| **ConvNeXt** | ViT design choices (GELU, LayerNorm, 7×7 DWConv, 4×4 stride-4 patchify stem) applied to pure CNNs |
| **EfficientNetV2** | Fused-MBConv (architecture) + progressive learning (training schedule) = faster training, same accuracy |
| **NFNet** | No BatchNorm needed — Scaled Weight Standardization + scaled residuals + AGC achieves SOTA |
| **ResNeXt / WRN** | Cardinality and width as orthogonal scaling axes to depth |
| **IoU / AP / mAP** | AP is the area under the per-class PR curve; mAP averages over **classes** (COCO also over 10 IoU thresholds) — **never over images** |
| **Faster R-CNN** | Backbone + FPN → RPN proposals from anchors → per-proposal classification and box refinement |
| **RoIAlign** | Drop the two quantization steps of RoI Pooling and bilinearly sample instead — what `MultiScaleRoIAlign` actually runs |
| **YOLO (v5 → YOLO26)** | Anchor-free decoupled head from v8; **NMS-free only from v10 onward**, standard in YOLO26 (which also drops DFL) |
| **Mask R-CNN** | Faster R-CNN + a per-RoI FCN mask branch, decoupled from classification, made possible by RoIAlign |
| **EfficientDet** | BiFPN with learnable weights = best multi-scale feature fusion |
| **SE / CBAM** | Channel + spatial attention → small overhead, meaningful gains |
| **1D CNNs** | Same CNN principles apply to temporal/signal data |
| **Transfer Learning** | Always start from pretrained weights; match the pretraining transform; use differential LR |
| **Benchmarks** | Compare recipes, not only architectures — a plain ResNet-50 gains 76.1 → 78.8 from the training recipe alone |



---

<a name='7-references'></a>
# 🔗 References
- Sergey Ioffe & Christian Szegedy (2015). [Batch Normalization: Accelerating Deep Network Training by Reducing Internal Covariate Shift](https://arxiv.org/pdf/1502.03167)
- Santurkar et al. (2018). [How Does Batch Normalization Help Optimization?](https://arxiv.org/abs/1805.11604) — NeurIPS; shows internal covariate shift is *not* why BN works (see §1.1).
- Wu & He (2018). [Group Normalization](https://arxiv.org/pdf/1803.08494) — ECCV; discusses small-batch limits of BN and GN’s channel-group statistics.
- Liu et al. (2022). [A ConvNet for the 2020s](https://arxiv.org/abs/2201.03545)
- Woo et al. (2023). [ConvNeXt V2: Co-designing and Scaling ConvNets with Masked Autoencoders](https://arxiv.org/pdf/2301.00808)
- Tan & Le (2021). [EfficientNetV2: Smaller Models and Faster Training](https://arxiv.org/abs/2104.00298)
- Brock et al. (2021). [High-Performance Large-Scale Image Recognition Without Normalization](https://arxiv.org/abs/2102.06171)
- Xie et al. (2017). [Aggregated Residual Transformations for Deep Neural Networks (ResNeXt)](https://arxiv.org/abs/1611.05431)
- Hendrycks & Gimpel (2016). [Gaussian Error Linear Units (GELUs)](https://arxiv.org/abs/1606.08415)
- Ramachandran et al. (2017). [Searching for Activation Functions](https://arxiv.org/abs/1710.05941) — Swish / SiLU.
- Girshick (2015). [Fast R-CNN](https://arxiv.org/abs/1504.08083) — introduces RoI Pooling.
- Ren et al. (2015). [Faster R-CNN: Towards Real-Time Object Detection with Region Proposal Networks](https://arxiv.org/abs/1506.01497)
- He et al. (2017). [Mask R-CNN](https://arxiv.org/abs/1703.06870) — RoIAlign and the decoupled mask branch.
- Lin et al. (2014). [Microsoft COCO: Common Objects in Context](https://arxiv.org/abs/1405.0312) — and the reference metric implementation, [`cocoeval.py`](https://github.com/cocodataset/cocoapi/blob/master/PythonAPI/pycocotools/cocoeval.py).
- Ultralytics docs. [YOLO11](https://docs.ultralytics.com/models/yolo11/) · [YOLO26](https://docs.ultralytics.com/models/yolo26/)
- Tan et al. (2020). [EfficientDet: Scalable and Efficient Object Detection](https://arxiv.org/abs/1911.09070)
- Hu et al. (2018). [Squeeze-and-Excitation Networks](https://arxiv.org/abs/1709.01507)
- Woo et al. (2018). [CBAM: Convolutional Block Attention Module](https://arxiv.org/abs/1807.06521)
- Li, J.; Wu (2024). [FSNB-YOLOV8: Improvement of Object Detection Model for Surface Defects Inspection in Online Industrial Systems](https://pdfs.semanticscholar.org/d27e/527d922d831fd791fc9345d07e2ea521564a.pdf)